In [51]:
from z3 import *
import re
from copy import deepcopy
# Define System E Language
PointSort = DeclareSort('Point')
LineSort = DeclareSort('Line')
CircleSort = DeclareSort('Circle')
RightAngle = RealVal(90)
bet = Function('bet', PointSort, PointSort, PointSort, BoolSort())
onLine = Function('onLine', PointSort, LineSort, BoolSort())
onCircle = Function('onCircle', PointSort, CircleSort, BoolSort())
inside = Function('inside', PointSort, CircleSort, BoolSort())
center = Function('center', PointSort, CircleSort, BoolSort())
sameside = Function('sameside', PointSort, PointSort, LineSort, BoolSort())
# Intersection Predicates
interLL = Function('interLL', LineSort, LineSort, BoolSort())
interLC = Function('interLC', LineSort, CircleSort, BoolSort())
interCC = Function('interCC', CircleSort, CircleSort, BoolSort())
# --- Magnitudes / Metrics (Return RealSort) ---
Segment = Function('Segment', PointSort, PointSort, RealSort())
Angle = Function('Angle', PointSort, PointSort, PointSort, RealSort())
Area = Function('Area', PointSort, PointSort, PointSort, RealSort())
# --- Constant Variables ---
A, B, C, D, E,F,G,H,I,J,K,L,M,N,P,Q,R,S = Consts('A B C D E F G H I J K L M N P Q R S', LineSort)
a,b,c,d,e,f,g,h,i,j,k,l,m,n,o,p,q,w,x,y,z = Consts('a b c d e f g h i j k l m n o p q w x y z', PointSort)
α,β,γ,δ,ε,ζ,η,θ,ι,κ,λ,μ,ω = Consts('α β γ δ ε ζ η θ ι κ λ μ ω', CircleSort)
## Const of Sort instance Initialiser
Point = lambda name: Const(name, PointSort)
Line = lambda name: Const(name, LineSort)
Circle = lambda name: Const(name, CircleSort)
## Functional object
line = Function('line', PointSort, PointSort, LineSort)
#def line(a: PointSort, b: PointSort): return Const(a.decl().name() + b.decl().name(), LineSort)  # deprecated
#ab = Line('ab'); bc = Line('bc'); ac = Line('ac'); ad = Line('ad')
alphabet = 'abcdefghijklmnopqwxyz'
for p1 in alphabet:    # Set up double lowercase line variables
    for p2 in alphabet:
        if p1 != p2:
            line_name = f"{p1}{p2}"
            # Fetch from globals ONLY if it exists and is a valid Z3 Point
            val1 = globals().get(p1)
            val2 = globals().get(p2)
            pt1 = val1 if (is_expr(val1) and val1.sort() == PointSort) else Const(p1, PointSort)
            pt2 = val2 if (is_expr(val2) and val2.sort() == PointSort) else Const(p2, PointSort)
            globals()[line_name] = line(pt1, pt2)
# --- Syntactic sugars ---
diffside = lambda a,b,L:  And(Not(onLine(a, L)), Not(onLine(b, L)), Not(sameside(a, b, L)))
outside = lambda a,α: And(Not(inside(a,α)), Not(onCircle(a,α)))
points = lambda *args: Or(Distinct(*args), Not(Distinct(*args)))   # A tautology for introducting points in premise
cong =  Function('cong', PointSort, PointSort, PointSort, PointSort, PointSort, PointSort, BoolSort())
coll = Function('coll', PointSort, PointSort, PointSort, BoolSort())
triangle = lambda a,b,c: Not(coll(a,b,c))       # Non-degenerate triangle
insideT = lambda p,a,b,c: And(sameside(p,a,line(b,c)), sameside(p,b,line(a,c)), sameside(p,c,line(a,b)) )   # Inside triangle abc
interior = lambda p,a,b,c: And( sameside(p,a,line(b,c)), sameside(p,c,line(a,b))  )     # point inside angle
sameray = lambda a,b,o: And( a!=o, b!=o, Or( a == b, bet(o,a,b), bet(o,b,a) ))      # two points on same ray w.r.t. vertex
inter1pt = lambda L,M: And( interLL(L,M), L != M )             # Two lines intersect at exactly one point
# sameside4 = Function('sameside4', PointSort, PointSort, PointSort, PointSort, BoolSort())


## ======== Axioms ==========
AXIOMS = { ## General Rules
'G1': ForAll([a,b,L,M], Implies( And(a != b, onLine(a,L), onLine(a,M), onLine(b,L), onLine(b,M)), L == M )), #兩點皆在線和線，則線線本同
'G2': ForAll([a,b,α,β], Implies( And(center(a,α), center(b,β)) , a == b )),  #點點忠圓則本同
'G3': ForAll([a,α], Implies( center(a,α) , inside(a,α) )),           #圓心入圓
'G4': ForAll([a,α], Implies( inside(a,α) , Not(onCircle(a,α)) )),    #圓內點不在圓
 ## Betweenness
'B1': ForAll([a,b,c], Implies( bet(a,b,c) , And( bet(c,b,a), Distinct(a,b,c), Not(bet(b,a,c)) )  )),#排則三點各異，首尾可換但首間不可換
'B2': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(a,L), onLine(b,L)), onLine(c,L)  )),   #線過排首兩點，則也過尾點
'B3': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(a,L), onLine(c,L)), onLine(b,L)  )),   #線過排首尾點，則也過間點
'B4': ForAll([a,b,c,d], Implies( And(bet(a,b,c), bet(a,d,b)), bet(a,d,c) )),    #排首兩之間點在首尾之間
'B5': ForAll([a,b,c,d], Implies( And(bet(a,b,c), bet(b,c,d)), bet(a,b,d) )),  #排尾兩是另排首兩，則間點在排首另排尾之間
'B6': ForAll([a,b,c,L], Implies( And(Distinct(a,b,c), onLine(a,L), onLine(b,L), onLine(c,L)),    #三點在線，必有一間點焉
                               And(AtLeast(bet(a,b,c),bet(b,a,c),bet(a,c,b), 1), AtMost(bet(a,b,c),bet(b,a,c),bet(a,c,b), 1)) )),
'B7': ForAll([a,b,c,d], Implies( And(bet(a,b,c), bet(a,b,d)), Not(bet(c,b,d)) )),   #兩排共首兩，則間點不在尾們之間
 ## Samesideness
'S1': ForAll([a,L], Implies( Not(onLine(a,L)), sameside(a,a,L) )),  #不在線點與己同邊
'S2': ForAll([a,b,L],  sameside(a,b,L) == sameside(b,a,L) ),   #點點同線邊可交換
'S3': ForAll([a,b,L], Implies( sameside(a,b,L), Not(onLine(a,L)) )),  #點點同線邊不在線
'S4': ForAll([a,b,c,L], Implies( And(sameside(a,b,L), sameside(a,c,L)), sameside(b,c,L) )),  #點點共點同線邊，則點點也同線邊
'S5': ForAll([a,b,c,L], Implies( And(Not(onLine(a,L)), Not(onLine(b,L)), Not(onLine(c,L)), Not(sameside(a,b,L)) ), 
                                 Xor(sameside(a,c,L),sameside(b,c,L)) )),   #點點點選線邊站，點點各一邊，則餘點必與點同邊
 ## Pasch axioms
'P1': ForAll([a,b,c,L], Implies( And(bet(a,b,c), sameside(a,c,L)), sameside(a,b,L)  )),   #排首尾同線邊，則首兩也同邊
'P2': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(a,L), Not(onLine(b,L))), sameside(b,c,L)  )),  #排首在線且間點不在線，則間尾同邊
'P3': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(b,L)), Not(sameside(a,c,L)) )),   #排間點在線，則首尾不同邊
'P4': ForAll([a,b,c,L,M], Implies( And(L!=M, onLine(b,L), onLine(b,M), onLine(a,M), onLine(c,M), Distinct(a,b,c), Not(sameside(a,c,L))), 
                                   bet(a,b,c) )),   #兩線有交點，另兩點皆在甲線且不同乙線邊，則交點為間點
## SKIPPED Triple incidence axioms FOR NOW
 ## Circles rules
'C1': ForAll([a,b,c,L,α], Implies( And(onLine(a,L),onLine(b,L),onLine(c,L), inside(a,α),onCircle(b,α),onCircle(c,α), b!=c), 
                                   bet(b,a,c) )),   #三點在線，其一入圓，餘兩點在圓，則入圓者在兩點之間
'C2': ForAll([a,b,c,α], Implies( And(Or(inside(a,α),onCircle(a,α)), Or(inside(b,α),onCircle(b,α)), bet(a,c,b)  ), 
                                   inside(c,α) )),   #排首尾皆在圓盤，則間點入圓
'C3': ForAll([a,b,c,α], Implies( And(Or(inside(a,α),onCircle(a,α)), Not(inside(c,α)), bet(a,c,b)  ), 
                                   And(Not(inside(b,α)), Not(onCircle(b,α))) )),   #排首在圓盤，間點不入圓，則尾出圓
'C4': ForAll([a,b,c,d,L,α,β], Implies( And(α!=β, c!=d, onCircle(c,α), onCircle(d,α), onCircle(c,β), onCircle(d,β), 
                                         center(a,α), center(b,β), onLine(a,L), onLine(b,L)   ), 
                                   Not(sameside(c,d,L)) )),    #兩圓交之兩點不同過各圓心之線邊
 ## Intersection rules
'I1': ForAll([a,b,L,M], Implies( And(Not(sameside(a,b,L)),Not(onLine(a,L)),Not(onLine(b,L)), onLine(a,M), onLine(b,M)  ), 
                                   interLL(L,M) )),   #隔開兩點之線必交過兩點之線
'I2': ForAll([a,b,L,α], Implies( And(Or(inside(a,α),onCircle(a,α)), Or(inside(b,α),onCircle(b,α)), 
                                     Not(sameside(a,b,L)),Not(onLine(a,L)),Not(onLine(b,L)) ), 
                                   interLC(L,α) )),   #隔開圓盤兩點之線交圓
'I3': ForAll([a,L,α], Implies( And(inside(a,α), onLine(a,L),   ), interLC(L,α) )),   #過圓內點之線交圓
'I4': ForAll([a,b,α,β], Implies( And(onCircle(a,α), Or(inside(b,α),onCircle(b,α)), inside(a,β), Not(Or(inside(b,β),onCircle(b,β))) ), 
                                   interCC(α,β) )),   #過圓內點並包含圓外點之另圓交圓
'I5': ForAll([a,b,α,β], Implies( And(onCircle(a,α), inside(b,α), inside(a,β), onCircle(b,β) ), 
                                   interCC(α,β) )),   #兩圓互過對方內點則相交
 ## Metric rules
'M1': ForAll([a,b], (Segment(a,b) == RealVal(0)) == (a == b) ),   #段長為零即端本同
'M2': ForAll([a,b], (Segment(a,b) >= RealVal(0)) ),  #段長非負
'M3': ForAll([a,b], Segment(a,b) == Segment(b,a) ),  #段端可互換
'M4': ForAll([a,b,c], Implies(And(a!=b, b!=c), Angle(a,b,c)==Angle(c,b,a)) ),  #角端不同頂，角端可互換
'M5': ForAll([a,b,c], And(RealVal(0) <= Angle(a,b,c), Angle(a,b,c) <= RealVal(180)) ),  #角度介乎零至一八零
'M6': ForAll([a,b], Area(a,a,b) == 0 ),  #兩角形面積為零
'M7': ForAll([a,b,c], Area(a,b,c) >= 0 ),   #面積非負
'M8': ForAll([a,b,c], And(Area(a,b,c) == Area(c,a,b), Area(c,a,b) == Area(a,c,b)) ),  #置換三角形頂點不變面積
'M9': ForAll([a,b,c,d,e,f], Implies(And(Segment(a,b)==Segment(d,e),Segment(b,c)==Segment(e,f), Segment(c,a)==Segment(f,d),
                                 Angle(a,b,c)==Angle(d,e,f), Angle(b,c,a)==Angle(e,f,d), Angle(c,a,b)==Angle(f,d,e) ),
                             Area(a,b,c) == Area(d,e,f) )),    #兩三角形對應邊角相等，則面積相等
 ## Diagram-segment transfer
'DS1': ForAll([a,b,c], Implies(bet(a,b,c), Segment(a,b) + Segment(b,c) == Segment(a,c) )),   #排首尾段長是首間間尾段長之和
'DS2': ForAll([a,b,c,α,β], Implies(And(center(a, α), center(a, β), onCircle(b, α), onCircle(c, β), Segment(a, b) == Segment(a, c)), 
                         α == β)),   #半徑相等的同心圓本同
'DS3': ForAll([a,b,c,α], Implies(And(center(a, α), onCircle(b, α)),   #惟點在圓時點心段長等於半徑
                                 (Segment(a, c) == Segment(a, b)) == onCircle(c, α) )),
'DS4': ForAll([a,b,c,α], Implies(And(center(a, α), onCircle(b, α)),   #惟點入圓時點心段長小於半徑
                                  (Segment(a, c) < Segment(a, b)) == inside(c, α) )),
 ## Diagram-angle transfer
'DA1': ForAll([a,b,c,L], Implies(And(a != b, a != c, onLine(a, L), onLine(b, L)),#三點成角，頂和一端在線，則惟另端在線且頂非於兩端之間時，角度為零
                             And(onLine(c, L), Not(bet(b, a, c))) == (Angle(b, a, c) == RealVal(0)) )),
'DA2': ForAll([a,b,c,d,L,M],   #兩線各過角端交於角頂，則惟有另點丁在角裏時，角度為端頂丁和丁頂另端之和
                 Implies(And(onLine(a, L), onLine(a, M), onLine(b, L), onLine(c, M), 
                             a != b, a != c, Not(onLine(d, L)), Not(onLine(d, M)), L != M),
                         (Angle(b, a, c) == Angle(b, a, d) + Angle(d, a, c)) == And(sameside(b, d, M), sameside(c, d, L)))),
'DA3': ForAll([a, b, c, d, L],   #兩鄰角相等惟皆直角時
                 Implies(And(onLine(a, L), onLine(b, L), bet(a, c, b), Not(onLine(d, L))),
                         (Angle(a, c, d) == Angle(d, c, b)) == (Angle(a, c, d) == RealVal(90)))),
'DA4': ForAll([a, b, c, e, f,L,M],    #兩角共頂點，角邊各共線，各頂和端同向，則兩角度相等
                 Implies(And(onLine(a,L),onLine(b,L),onLine(e,L),onLine(a,M),onLine(c,M),onLine(f,M), b != a, e != a, c != a, f != a, Not(bet(b, a, e)), Not(bet(c, a, f))),
                         Angle(b, a, c) == Angle(e, a, f))),
'DA5': ForAll([a, b, c, d, e, L, M, N],   #同旁內角和小於平角，則兩被截線相交於角端之側
                 Implies(And(onLine(a, L), onLine(b, L), onLine(b, M), onLine(c, M), onLine(c, N), onLine(d, N), 
                             b != c, sameside(a, d, M), (Angle(a, b, c) + Angle(b, c, d)) < RealVal(180)),
                         And(interLL(L, N), Implies(And(onLine(e, L), onLine(e, N)), sameside(e, a, M))) )),
 ## Diagram-area transfer
'DR1': ForAll([a, b, c, L],  # 兩點在線，與另點成形，惟該點也在線時面積為零
        Implies(And(onLine(a, L), onLine(b, L), a != b), (Area(a, b, c) == RealVal(0)) == onLine(c, L))),
'DR2': ForAll([a, b, c, d, L],  #三點在線，另點不在，則惟三點成排時，首尾另面積為首間另與另間尾之和
                    Implies(And(onLine(a, L), onLine(b, L), onLine(c, L), Distinct(a, b, c), Not(onLine(d, L))),
                         bet(a, c, b) == (Area(a, c, d) + Area(d, c, b) == Area(a, d, b)))),

 ## Triangle Congruence
'CG': ForAll([a,b,c,d,e,f], cong(a,b,c,d,e,f) == And(Segment(a,b)==Segment(d,e),Segment(b,c)==Segment(e,f), Segment(c,a)==Segment(f,d),
                 Angle(a,b,c)==Angle(d,e,f), Angle(b,c,a)==Angle(e,f,d), Angle(c,a,b)==Angle(f,d,e))  ),  # Triangle Congreunce def

'L0': ForAll([a,b,c], coll(a,b,c) == Or( bet(a,b,c), bet(b,c,a), bet(c,a,b), a==b, b==c, a==c )),   # Collinearity def
'L4': ForAll([a,b,c,L], Implies( And(onLine(a,L),onLine(b,L),onLine(c,L)), coll(a,b,c) )),   # Collinearity def 2

}
## ======= CONSTRUCTION RULES ======== 
CONSTRUCTION_RULES = {  ## === Points ===     each rule consists of [ args => premises, args => conclusions ]
# CP0: 有點同另點  ∃ a 使 a = b
'CP0': [ lambda b: BoolVal(True),     
        lambda a, b: a == b ],
# CP1: 有點異另點  ∃ a 使 a ≠ b
'CP1': [ lambda b: BoolVal(True),     
        lambda a, b: a != b ],
# CP2: 線上有點  ∃ a 使 a on L
'CP2': [ lambda L: BoolVal(True),     
        lambda a, L: onLine(a, L) ],
# CP2ne: 線上有點  ∃ a 使 a on L and a ≠ b
'CP2ne': [ lambda b,L: BoolVal(True),     
        lambda a,b,L: And(onLine(a, L), a != b) ],
# CP3: 兩點之間有點  b c on L, b ≠ c ⊢ ∃a 使 a on L ∧ bet(b,a,c)
'CP3': [ lambda b, c, L: And(onLine(b, L), onLine(c, L), b != c ),
         lambda a, b, c, L: And(onLine(a, L), bet(b, a, c)) ],
# CP4: 兩點之後有點  b, c on L, b ≠ c ⊢ a on L ∧ bet(b,c,a)
'CP4': [ lambda b, c, L: And(onLine(b, L), onLine(c, L), b != c),
         lambda a, b, c, L: And(onLine(a, L), bet(b, c, a)) ],
# CP5: 有點同邊不在線點 (¬(b on L) ⊢ a, b sameside L)
'CP5': [ lambda b, L: Not(onLine(b, L)),
         lambda a, b, L: sameside(a, b, L) ],
# CP6: 有點異邊不在線點 (¬(b on L) ⊢ a, b diffside L)
'CP6': [ lambda b, L: Not(onLine(b, L)),
         lambda a, b, L: diffside(a, b, L) ],
# CP7: 圓上有點  ∃a 使 a on α
'CP7': [ lambda α: BoolVal(True),
         lambda a, α: onCircle(a, α) ],
# CP8: 圓內有點  ∃a 使 a inside α 
'CP8': [ lambda α: BoolVal(True),
         lambda a, α: inside(a, α) ],
# CP9: 圓外有點  ∃a 使 a outside α
'CP9': [ lambda α: BoolVal(True),
         lambda a, α: And(outside(a, α)) ],
## ===== Lines and circles =====
# LC1: 兩點有一線過 (a ≠ b ⊢ ∃L 使 a on L ∧ b on L)
'LC1': [ lambda a, b: a != b,
         lambda L, a, b: And(onLine(a, L), onLine(b, L)) ],
# LC2: 有圓中心點過另點 (a ≠ b ⊢ ∃ α 使 a center α ∧ b on α)
'LC2': [ lambda a, b: a != b,
         lambda α, a, b: And(center(a, α), onCircle(b, α)) ],
## ===== Intersections =====
# CI1: 線交線有點  L inter M ⊢ ∃a 使 a on L ∧ a on M
'CI1': [ lambda L, M: interLL(L, M),
         lambda a, L, M: And(onLine(a, L), onLine(a, M)) ],
# CI2: 圓交線有點  α inter L ⊢ ∃a 使 a on α ∧ a on L
'CI2': [ lambda L, α: interLC(L, α),
         lambda a, L, α: And(onCircle(a, α), onLine(a, L)) ],
# CI3: 圓交線有兩點  α inter L ⊢ ∃a b 使a b on α ∧ a b on L ∧ a ≠ b
'CI3': [ lambda L, α: interLC(L, α),
         lambda a, b, L, α: And(onCircle(a, α), onLine(a, L), onCircle(b, α), onLine(b, L), a != b) ],
# CI4: 線過圓內外兩點，則兩點之間有交點  b inside α, b on L, c outside α, c on L  ⊢ ∃a 使a on α ∧ a on L ∧bet(b,a,c)
'CI4': [ lambda b, c, L, α: And(inside(b, α), onLine(b, L), outside(c, α), onLine(c, L)),
         lambda a, b, c, L, α: And(onCircle(a, α), onLine(a, L), bet(b, a, c)) ],
# CI5: 線過點和圓內另點，則兩點之後有交點  b inside α, b on L, c ≠ b, c on L ⊢ ∃a 使a on α ∧ a on L ∧ bet(a,b,c)
'CI5': [ lambda b, c, L, α: And(inside(b, α), onLine(b, L), c != b, onLine(c, L)),
         lambda a, b, c, L, α: And(onCircle(a, α), onLine(a, L), bet(a, b, c)) ],
# CI6: 圓交圓有點  α inter β ⊢ ∃a 使 a on α ∧ a on β
'CI6': [ lambda α, β: interCC(α, β),
         lambda a, α, β: And(onCircle(a, α), onCircle(a, β)) ],
# CI7: 圓交圓有兩點  α inter β ⊢ ∃a b 使 a on α β ∧ b on α β ∧ a ≠ b
'CI7': [ lambda α, β: interCC(α, β),
         lambda a, b, α, β: And(onCircle(a, α), onCircle(a, β), onCircle(b, α), onCircle(b, β), a != b) ],
# CI8: 圓交圓有點在過各心之線的點同邊  α inter β, c center α, d center β. c d on L, ¬(b on L) ⊢ ∃a 使a on α ∧ a on β ∧ a b sameside L
'CI8': [ lambda b, c, d, L, α, β: And(interCC(α, β), center(c, α), center(d, β), onLine(c, L), onLine(d, L), Not(onLine(b, L))),
         lambda a, b, c, d, L, α, β: And(onCircle(a, α), onCircle(a, β), sameside(a, b, L)) ],
# CI9: 圓交圓有點在過各心之線的點異邊  α inter β, c center α, d center β, c d on L, ¬(b on L) ⊢ ∃a 使a on α ∧ a on β ∧ a b diffside L
'CI9': [ lambda b, c, d, L, α, β: And(interCC(α, β), center(c, α), center(d, β), onLine(c, L), onLine(d, L), Not(onLine(b, L))),
         lambda a, b, c, d, L, α, β: And(onCircle(a, α), onCircle(a, β), diffside(a, b, L)) ],

# SP: Superposition principle
'SP': [ lambda a,b,c,d,g,h: And( Distinct(a,b,c), d!=g, Not(onLine(h,line(d,g))) ),
        lambda x,y,z,a,b,c,d,g,h: And( x==d, sameray(y,g,d), sameside(z,h,line(d,g)), cong(x,y,z,a,b,c)   )  ]
    # Segment(a,b)==Segment(x,y), Segment(a,c)==Segment(x,z), Segment(b,c)==Segment(y,z),
## EXTRAS (for convenience)
# SS: sameside4 to sameside
# 'SS': [ lambda d,a,b,c: sameside4(d,a,b,c),
#          lambda L,d,a,b,c: And( onLine(b,L),onLine(c,L), sameside(d,a,L) ) ],
}
## ======= DEFAULT AXIOMS (no need to supply reason) =======
DEFAULTS = {  
'B1part': ForAll([a,b,c], Implies( bet(a,b,c) , And( bet(c,b,a), Distinct(a,b,c), Not(bet(b,a,c)) )  )),
'S2': AXIOMS['S2'],
'M0': ForAll([a], Segment(a,a) == RealVal(0)  ),    # M0: Segment(a,a) = 0      // 單點無長
'M2': AXIOMS['M2'],    # M2: Segment(a,b) ≥ 0      // 段長非負
'M3': AXIOMS['M3'],    # M3: Segment(a,b) = Segment(b,a)    // 段端可互換
'M4': AXIOMS['M4'],    # M4: a ≠ b, b ≠ c ⊢ ∠(a,b,c) = ∠(c,b,a)   // 角端不同頂，角端可互換
'M5': AXIOMS['M5'],    # M5: 0 ≤ ∠(a,b,c) ≤ 180   // 角度介乎零至一八零
'M6': AXIOMS['M6'],    # M6: Area(a,a,b) = 0   // 兩角形面積為零
'M7': AXIOMS['M7'],    # M7: Area(a,b,c) ≥ 0   // 面積非負
'L0': AXIOMS['L0'],    # L0: coll(a,b,c) == Or( bet(a,b,c), bet(b,c,a), bet(c,a,b), a==b, b==c, a==c )   Collinearity def
'L1': ForAll([a,b,c], Implies( bet(a,b,c), coll(a,b,c) )),  # Collinearity (for convenience) #排則共線
'L2': ForAll([a,b,c],  coll(a,b,c) == coll(c,b,a) ),   # Terminals can swap
'L3': ForAll([a,b,c],  coll(a,b,c) == coll(b,a,c) ),   # First two can swap
#'L4': AXIOMS['L4'],    # L4: a b c on L ⊢ coll(a,b,c)
'LL': ForAll([L,M], interLL(L,M) == interLL(M,L) ),    # inter commute
'CG': AXIOMS['CG'],    # Corresponding sides & angles of cong triangles
'l0': ForAll([a,b], line(a,b) == line(b,a) ),    # Line object commute
'l1': ForAll([a,b], And( onLine(a,line(a,b)), onLine(b,line(a,b)) ), patterns=[line(a, b)]),     # Implicit point onLine rule
#'SS': ForAll([a,b,c,d,L], Implies( And(onLine(b,L),onLine(c,L), sameside(a,d,L) ), sameside4(a,d,b,c) )  ),
#'IT': ForAll([a,b,c,d], insideTri(d,a,b,c) == And( sameside4(d,a,b,c), sameside4(d,b,a,c), sameside4(d,c,a,b) )), # inside triangle def
}


In [52]:
class Proof:
    def __init__(self, premises, goal, solver: Solver = None):
        if solver is None:
            solver = Solver(); solver.set("timeout", 2000)
        self.solver = solver
        self.axioms = AXIOMS
        self.z3_vars = {}    # Store Z3 variables, indexed by str
        self.z3_vars_hist = []    # For cases variables revertion
        self.context: [ExprRef] = [ ]    # Stores premises and any valid formulas proved
        self.context_hist = []    # For cases context revertion
        self.implicits: set[ExprRef] = set()    # Stores implicit rules generated by functional objects
        self.goal = 'undefined'
        self.premises = 'undefined'
        self.premise_vars = set()
        self.goal_vars = set()
        self.goalonly_vars = set()   # = goal_vars - premise_vars
        self.loadContextAndGoal(premises, goal)
        ## For storing tactics info
        self.cases_layer = 0   # Each case split will +1 until resolved
        self.cases_stack: [[ExprRef]] = []
        self.current_case = None
        self.current_case_nth = None
        self.current_or_expr = None

    def loadContextAndGoal(self, premises: list[ExprRef], goal: ExprRef):  # Called at init
        # 0. Shallow copy premises
        self.premises = list(premises)
        self.premise_vars = set()
        self.goal_vars = set()
        self.goalonly_vars = set()
        self.goal = goal
        # 1. Load explicit premises into context & solver
        for prem in premises:
            if prem is not None:
                self.add(prem)
        # 3. Collect variables from premises
        for expr in premises:
            expr_vars = self.collect_vars(expr)
            self.premise_vars.update(expr_vars)
            for var_obj in expr_vars:
                name = var_obj.decl().name()
                if name not in self.z3_vars:
                    self.z3_vars[name] = var_obj
    
        # 4. Safely add implicit parsed rules to context
        # parsed_rule = None
        # if parsed_rule_list:
        #     parsed_rule = parsed_rule_list[0] if len(parsed_rule_list) == 1 else And(parsed_rule_list)
        #     if hasattr(parsed_rule, 'children') and parsed_rule.children(): 
        #         self.add(parsed_rule)
    
        # Print premises and goal for display
        for i, t in enumerate(premises):
            print(f"{i+1}. {t}", end='\n' if i < len(premises) - 1 else '')
        print(f'    / {self.goal}')
        # if parsed_rule is not None and hasattr(parsed_rule, 'children') and parsed_rule.children(): 
        #     print(f"{len(self.context)}. {parsed_rule}\t(imp)")
        print('-' * 20)
    
        # 5. Collect Goal Variables
        self.goal_vars.update(self.collect_vars(goal))
    
        # 6. Identify all known point names from premises
        # known_point_chars = set()
        # for var_obj in self.premise_vars:
        #     sort_str = str(var_obj.sort())
        #     if sort_str == 'Point':
        #         known_point_chars.add(var_obj.decl().name())
        #     elif sort_str == 'Line' and len(var_obj.children()) == 2:
        #         # Extract point arguments from line(p1, p2)
        #         for child in var_obj.children():
        #             known_point_chars.add(child.decl().name())
    
        # # 7. Find "Dependent Lines" in the goal and promote them to premise_vars
        # dependent_lines = set()
        # for var_obj in self.goal_vars:
        #     sort_str = str(var_obj.sort())
        #     # Safe check for line(a, b) application terms in the goal
        #     if sort_str == 'Line' and len(var_obj.children()) == 2:
        #         p1_name = var_obj.children()[0].decl().name()
        #         p2_name = var_obj.children()[1].decl().name()
        #         if p1_name in known_point_chars and p2_name in known_point_chars:
        #             dependent_lines.add(var_obj)
    
        # # Move dependent lines out of "goal-only" category
        # self.premise_vars.update(dependent_lines)
        # for var_obj in dependent_lines:
        #     name = var_obj.decl().name()
        #     if name not in self.z3_vars:
        #         self.z3_vars[name] = var_obj
    
        # 8. Compute true goal-only variables (construction targets)
        self.goalonly_vars = self.goal_vars - self.premise_vars
    
        # 9. Helper sorting key function
        def sort_key(t):
            name = t.decl().name() if hasattr(t, 'decl') else str(t)
            first_char = name[0] if name else ''
            priority = 0 if 'a' <= first_char <= 'z' else 1 if 'A' <= first_char <= 'Z' else 2
            return (priority, name)
    
        self.sorted_premise_vars = sorted(list(self.premise_vars), key=sort_key)
        self.sorted_goalonly_vars = sorted(list(self.goalonly_vars), key=sort_key)
    
        # 10. Load default base theory axioms
        for rule in DEFAULTS.values():
            self.solver.add(rule)
    
        return

    # Helper: Native AST Variable Harvester (No Regex, No eval!)
    def collect_vars(self, expr) -> set[ExprRef]:
        if len(expr.children()) == 0:   # If it has no children, it's a leaf node
            if expr.decl().name() not in ['True', 'False']:  # Skip boolean literals like True/False
                return {expr}
            return set()
        found = set()
        for child in expr.children():  # Otherwise, recursively gather variables from all sub-expressions
            found.update(self.collect_vars(child))
        return found
    # Helper: add expression to both context and solver
    def add(self, expr):
        self.solver.add(expr); self.context.append(expr)
        
    # Helper: Recursively parse Z3 expressions (to add assisting rules like And(onLine(a,ab),onLine(b,ab))) for line(a,b))
    # Helper: Recursively parse Z3 expressions to extract implicit line rules (e.g., onLine(a, line(a, b)))
    def parse_expr(self, expr, is_first_time=False):
        rules = []
        # Guard against non-Z3 objects or expressions without children
        if not is_expr(expr):
            return rules
        # 1. Detect uninterpreted function calls like line(p1, p2)
        # Check if the expression evaluates to LineSort and has 2 point arguments
        if expr.sort() == LineSort and len(expr.children()) == 2:
            p1 = expr.children()[0]
            p2 = expr.children()[1]
            # Directly construct implicit incidence rule
            implicit_rule = And(onLine(p1, expr), onLine(p2, expr))
            rules.append(implicit_rule)
        # 2. Recursively scan all sub-expressions / children in the AST
        for child in expr.children():
            rules.extend(self.parse_expr(child, is_first_time))
        # 3. Deduplicate rules using Z3 AST structural equality (.eq)
        unique_rules = []
        for r in rules:
            if not any(r.eq(existing) for existing in unique_rules):
                unique_rules.append(r)
        return unique_rules
        
    # Use rule to construct a new variable from existing objs
    def construct(self, rule: str, new: SortRef, objs: list, debug=True):  
        targets = list(new) if isinstance(new, (list, tuple)) else [new]
        # 1. Check if ANY of the target variable names are already in use
        for t in targets:
            var_name = t.decl().name()
            if var_name in self.z3_vars:
                print(f"--> #{len(self.context)+1} construct{rule, new, objs}: Variable '{var_name}' is already in use!")
                return
        # 2. Extract rule logic formulas
        try:
            premises = CONSTRUCTION_RULES[rule][0](*objs)
            conclusion = CONSTRUCTION_RULES[rule][1](*targets, *objs)
        except KeyError:
            print(f"--> #{len(self.context)+1} construct{rule, new, objs}: Invalid construction rule '{rule}'!")
            return
        except TypeError as e:
            print(f"--> #{len(self.context)+1} construct{rule, new, objs}: Argument count mismatch: {e}")
            return
        except Exception as e:
            print(f"--> #{len(self.context)+1} construct{rule, new, objs}: {e}!")
            return
        # FIX 1: Use .eq() for Z3 AST containment check
        if any(conclusion.eq(c) for c in self.context):
            print(f"--> #{len(self.context)+1} construct{rule, new, objs}: You have constructed {conclusion} before. Try again.")
            return
        # 3. Check if all the rule's premises can be inferred from self.context
        self.solver.push()    # Make checkpoint
        self.solver.add(Not(premises))
        if self.solver.check() == sat:   # Can't construct as premises not met
            print(f"--> #{len(self.context)+1} construct{rule, new, objs}:", end=' ')
            print(f"Sorry, can't construct {new} as premises are not met: {premises}")
            # FIX 3 & 4: Safe sub-premise debugging without mutating proof state
            if debug and hasattr(premises, 'decl') and premises.decl().name() == 'and':
                print("  Missing sub-premises:")
                for prem in premises.children():
                    self.solver.push()
                    self.solver.add(Not(prem))
                    if self.solver.check() == sat:
                        print(f"    ❌ Not satisfied: {prem}")
                    self.solver.pop()
    
            self.solver.pop()
            return
        self.solver.pop()
        # 4. Success State Commitments
        self.add(conclusion)
        
        # Register the newly created variables to prevent future collisions
        for t in targets:
            self.z3_vars[t.decl().name()] = t
        print(f"{len(self.context)}. Constructed {new} where {conclusion} \t[{rule}]")
        return
    # Assert statements
    def have(self, fmla: ExprRef, rule: str=None, debug=False):    # Infer step
        self.solver.push()    # Make checkpoint
        if rule == 'sorry':   # CHEAT rule to allow to use EVERY AXIOM (but not constructions)
            for rue in AXIOMS:
                self.solver.add(AXIOMS[rue])
        elif rule is not None and rule not in DEFAULTS:
            try:
                self.solver.add( AXIOMS[rule] )   # Only allow this axiom in solver
            except KeyError:
                print(f"--> #{len(self.context)+1} have({fmla}, {rule}):", end=' ')
                print("Invalid reason!") 
            except Exception as e:
                print(f"--> #{len(self.context)+1} have({fmla}, {rule}): {e}!")
                return
                self.solver.pop(); return
        if fmla in self.context:
            print(f"--> #{len(self.context)+1} have({fmla}, {rule}):", end=' ')
            print(f"You have proved {fmla} before. Try again.")
            self.solver.pop(); return
        self.solver.add( Not(fmla) )   # Check if solver can infer the formula
        # 1. Get the raw list of implicit rules from the formula parser
        raw_rules = self.parse_expr(fmla, True)
        # 2. Bundle them up temporarily just to assist the solver's check phase
        if raw_rules:
            check_rule = raw_rules[0] if len(raw_rules) == 1 else And(raw_rules)
            if check_rule.children(): 
                self.solver.add(check_rule)
        result = self.solver.check()
        # --- FAILURE HANDLING & DEBUG BREAKDOWN ---
        if result == sat or result == unknown:    # Can't infer
            print(f"--> #{len(self.context)+1} have({fmla}, {rule}): Sorry, can't infer {fmla} from {rule}.", end='\n')
            if debug:
                print(f"\n[DEBUG] --- RULE ANTECEDENT DIAGNOSIS ({rule}) ---")
                self.solver.pop()   # Clear the Not(fmla) statement to evaluate premises cleanly
                
                if rule in AXIOMS:
                    import z3
                    from z3 import is_quantifier, is_var, substitute_vars
                    axiom = AXIOMS[rule]
                    body = axiom.body() if is_quantifier(axiom) else axiom
                    
                    if body.decl().name() == '=>':
                        premise_pat = body.children()[0]
                        conclusion_pat = body.children()[1]
                        
                        # 1. Core structural matcher
                        def match_expr(pattern, instance, subst):
                            if is_var(pattern):
                                idx = z3.Z3_get_index_value(pattern.ctx.ref(), pattern.as_ast())
                                if idx in subst:
                                    return str(subst[idx]) == str(instance)
                                subst[idx] = instance
                                return True
                            if pattern.decl().name() != instance.decl().name():
                                return False
                            
                            p_name = pattern.decl().name()
                            if p_name in ['and', 'or']:
                                p_children = list(pattern.children())
                                i_children = list(instance.children())
                                if len(p_children) != len(i_children): return False
                                temp_subst = dict(subst)
                                for p_child in p_children:
                                    found = False
                                    for i_child in i_children:
                                        backup = dict(temp_subst)
                                        if match_expr(p_child, i_child, temp_subst):
                                            i_children.remove(i_child)
                                            found = True
                                            break
                                        else:
                                            temp_subst = backup
                                    if not found: return False
                                subst.update(temp_subst)
                                return True
                            else:
                                if len(pattern.children()) != len(instance.children()): return False
                                return all(match_expr(p, i, subst) for p, i in zip(pattern.children(), instance.children()))

                        # 2. Deep Scanner: Collect ALL potential binding dictionaries from compound conclusions
                        def collect_bindings_from_target(pattern, target, current_subst):
                            results = []
                            subst = dict(current_subst)
                            if match_expr(pattern, target, subst):
                                results.append(subst)
                            
                            if pattern.decl().name() in ['and', 'or', 'xor', 'not', '=>']:
                                for child in pattern.children():
                                    results.extend(collect_bindings_from_target(child, target, current_subst))
                            return results

                        all_candidates = collect_bindings_from_target(conclusion_pat, fmla, {})
                        
                        # Deduplicate candidates
                        unique_candidates = []
                        seen_candidates = set()
                        for c in all_candidates:
                            t = tuple(sorted(c.items()))
                            if t not in seen_candidates:
                                seen_candidates.add(t)
                                unique_candidates.append(c)

                        num_vars = axiom.num_vars() if is_quantifier(axiom) else 0
                        clauses = premise_pat.children() if premise_pat.decl().name() == 'and' else [premise_pat]

                        # Helper to map de Bruijn indices back to friendly declared names
                        def get_var_name(idx):
                            if is_quantifier(axiom):
                                try: return axiom.var_name(num_vars - 1 - idx)
                                except: pass
                            return f"Var({idx})"

                        if False:  #not unique_candidates:
                            print("  [X] Error: The target formula does not match the conclusion of the rule at all.")
                        else:
                            print(f"  [*] Found {len(unique_candidates)} potential structural branch matches in the rule conclusion.")
                            for idx, initial_subst in enumerate(unique_candidates, 1):
                                print(f"\n  --- Match Interpretation #{idx} ---")
                                subst = dict(initial_subst)
                                
                                # 3. Context loop to resolve remaining variables
                                updated = True
                                while updated and len(subst) < num_vars:
                                    updated = False
                                    curr_len = len(subst)
                                    for clause in clauses:
                                        for fact in self.context:
                                            temp_subst = dict(subst)
                                            if match_expr(clause, fact, temp_subst):
                                                if len(temp_subst) > curr_len:
                                                    subst.update(temp_subst)
                                                    updated = True
                                                    curr_len = len(subst)
                                                    break
                                
                                # Display named bindings nicely
                                named_bindings = {get_var_name(k): v for k, v in subst.items()}
                                args = [subst.get(i, None) for i in range(num_vars)]
                                
                                if num_vars > 0 and None in args:
                                    missing_names = [get_var_name(i) for i in range(num_vars) if args[i] is None]
                                    print(f"    [!] Partial bindings resolved: {named_bindings}")
                                    print(f"    [X] Missing variables from context: {missing_names}")
                                    print(f"    (The engine cannot evaluate premises because these elements are completely unreferenced in your proof history.)")
                                else:
                                    print(f"    [+] Full bindings resolved successfully: {named_bindings}")
                                    print(f"    [*] Evaluating rule premises:")
                                    for clause in clauses:
                                        concrete_clause = substitute_vars(clause, *args) if num_vars > 0 else clause
                                        
                                        self.solver.push()
                                        self.solver.add(Not(concrete_clause))
                                        test_res = self.solver.check()
                                        self.solver.pop()
                                        
                                        if test_res == unsat:
                                            print(f"    [V] Premise SATISFIED: {concrete_clause}")
                                        else:
                                            print(f"    [X] Premise NOT SATISFIED: {concrete_clause} <--- Missing link!")
                    else:
                        print(f"  [*] Rule {rule} is not an implication. Cannot extract premises.")
                else:
                    print(f"  [*] Rule {rule} not found in AXIOMS.")
                print("[DEBUG] ---------------------------------------------\n")
                return
                
            self.solver.pop()   # Fallback cleanup
            return
            
        # Successful deduction step
        self.solver.pop()   # Remove the axiom used from solver 
        
        if not debug: 
            self.add(fmla)
            step_num = len(self.context)
            print(f"{step_num}. {fmla}\t[{rule}]")
            
            # 3. FILTER TIME: Compare the raw rules directly against your self.implicits set
            # new_rules = [r for r in raw_rules if r not in self.implicits]
            
            # # 4. Process and store only if there are genuinely unseen implicit rules
            # if new_rules:
            #     filtered_rule = new_rules[0] if len(new_rules) == 1 else And(new_rules)
                
            #     if filtered_rule not in self.context:
            #         self.add(filtered_rule)
                    
            #         # Core fix: add the unique raw rules themselves to the set, NOT their variables
            #         self.implicits.update(new_rules)
                    
            #         imp_step_num = len(self.context)
            #         print(f"{imp_step_num}. {filtered_rule}\t(imp)")
        else: 
            print(f"SATISFIED. {fmla}\t[{rule}]")
            # Clean handling for debug mode
            new_rules = [r for r in raw_rules if r not in self.implicits]
            if new_rules:
                filtered_rule = new_rules[0] if len(new_rules) == 1 else And(new_rules)
                print(f"SATISFIED. {filtered_rule}\t(imp)")

    def qed(self, prop_name: str = None):
        # 0. Check if solver is inside a case split
        if self.cases_layer > 0:
            print(f"❌ Still have {self.cases_layer} layer of unresolved cases.")
            return
        # 1. Check if any formula in self.context matches the goal
        for fmla in self.context:
            if fmla.eq(self.goal):
                print(f"🔥 Success! You proved {self.goal} is true.")
                if prop_name is None: return  # Don't record into AXIOMS / CONSTRUCTION_RULES if thm name is not specified
                # Make sure we have a single base premise expression to work with
                if isinstance(self.premises, list):
                    base_premise = self.premises[0] if len(self.premises) == 1 else And(self.premises)
                else:
                    base_premise = self.premises
    
                # Snapshot references to our original variables
                # orig_inputs = [ i for i in self.sorted_premise_vars if \
                #                 not(len(i.decl().name())==2 and i.decl().name()[0] in alphabet and i.decl().name()[1] in alphabet )]
                # 1. Identify point variables and dependent line variables
                point_names = {v.decl().name(): v for v in self.sorted_premise_vars if str(v.sort()) == 'Point'}
                orig_inputs = []
                orig_outputs = list(self.sorted_goalonly_vars)
                orig_goal = self.goal
                double_lowers = []
                for v in self.sorted_premise_vars:
                    v_name = v.decl().name()
                    if str(v.sort()) == 'Line' and len(v_name) == 2 and v_name[0] in point_names and v_name[1] in point_names:
                        double_lowers.append(v)
                    else:
                        orig_inputs.append(v)

                # 2. Closure helper that constructs fresh line constants for runtime points
                def build_line_subs(runtime_inputs):
                    # Map original point constants -> new runtime point constants
                    param_map = {orig_v.decl().name(): r_v for orig_v, r_v in zip(orig_inputs, runtime_inputs)}
                    line_subs = []
                    for dv in double_lowers:
                        v_name = dv.decl().name() # e.g. "ab"
                        p1 = param_map[v_name[0]] # runtime point for 'a'
                        p2 = param_map[v_name[1]] # runtime point for 'b'
                        
                        # Calls line(p1, p2) dynamically -> produces Const("p1p2", Line)
                        line_subs.append((dv, line(p1, p2)))
                    return line_subs
    
                # 2. Dynamic Closure for the Rule Premise
                def premise_closure(*runtime_inputs):
                    # Maps original proof inputs -> whatever new variables the user passes in
                    sub_pairs = list(zip(orig_inputs, runtime_inputs)) + build_line_subs(runtime_inputs)
                    return substitute(base_premise, *sub_pairs)
    
                # 3. Dynamic Closure for the Rule Conclusion
                def goal_closure(*runtime_args):
                    # runtime_args matches the execution signature: (*outputs, *inputs)
                    runtime_outputs = runtime_args[:len(orig_outputs)]
                    runtime_inputs = runtime_args[len(orig_outputs):]
                    
                    # Zip old variable references to new ones across the entire scope
                    old_vars = orig_outputs + orig_inputs
                    new_vars = list(runtime_outputs) + list(runtime_inputs)
                    sub_pairs = list(zip(old_vars, new_vars)) + build_line_subs(runtime_inputs)
                    
                    return substitute(orig_goal, *sub_pairs)
    
                # 4. Save the executable closures directly into AXIOMS/CONSTRUCTION_RULES
                if len(self.goalonly_vars) > 0:
                    # CASE A: Standard Construction Rule (Creates new points/lines/circles)
                    if prop_name not in CONSTRUCTION_RULES:
                        CONSTRUCTION_RULES[prop_name] = [premise_closure, goal_closure]
                        
                        pvars_str = ', '.join([t.decl().name() for t in self.sorted_premise_vars])
                        gvars_str = ', '.join([t.decl().name() for t in self.sorted_goalonly_vars])
                        print(f"💾 Added Construction Rule {prop_name}: [lambda {pvars_str}: {base_premise}, "
                              f"lambda {gvars_str}, {pvars_str}: {orig_goal}] into CONSTRUCTION RULES.")
                else:
                    # CASE B: Pure Deduction Theorem (No new variables, e.g., Prop 5)
                    # Build a universally quantified Z3 implication
                    if len(orig_inputs) > 0:
                        theorem_formula = ForAll(orig_inputs, Implies(base_premise, orig_goal))
                    else:
                        theorem_formula = Implies(base_premise, orig_goal)
                    
                    # Store it in a global dictionary for theorems/axioms
                    # (Make sure to define THEOREMS = {} globally alongside CONSTRUCTION_RULES)
                    AXIOMS[prop_name] = theorem_formula
                    
                    print(f"💾 Registered {prop_name} as a Universal Theorem in AXIOMS!")
                    print(f"   Form: {theorem_formula}")
            
                return True
    
        print(f"❌ You haven't proved {self.goal} yet.")
        return False
    ## ===== Proof Tactics =====
    # Helpers for saving self.context state
    def push(self):
        self.solver.push()                            # solver snapshot
        self.context_hist.append(list(self.context))  # context snapshot
        self.z3_vars_hist.append(dict(self.z3_vars))  # variables snapshot
        #print('scv(pushed):', self.solver, '|', self.context_hist, '|', self.z3_vars_hist)
    def pop(self):
        if not self.context_hist:
            raise ValueError("No checkpoint to pop!")
        self.solver.pop()                       # solver rolls back
        self.context = self.context_hist.pop()  # context rolls back to the exact same moment
        self.z3_vars = self.z3_vars_hist.pop()  # variables rolls back
        #print('scv(popped):', self.solver, '|', self.context_hist, '|', self.z3_vars_hist)
    # Proof by cases (Or-elim)
    def cases(self, or_expr: ExprRef):
        self.cases_layer += 1
        if or_expr.decl().name() != 'or':
            print(f"--> #{len(self.context)+1} case({or_expr}): Can't case split on non-Or expression.")
            return
        if not any(or_expr.eq(c) for c in self.context):
            print(f"--> #{len(self.context)+1} case({or_expr}): This Or expression is not in the context.")
            return
        self.current_or_expr = or_expr
        self.push()   # checkpoint trioth solver and context and z3_vars
        children = list(reversed(or_expr.children()))
        self.cases_stack.append(children )
        first_case = children.pop()
        self.current_case = first_case; self.current_cases_nth = 1
        self.add(first_case)   # Assume the first case and try to reach the goal
        print(f"{len(self.context)}.{self.current_cases_nth}. Assuming case {first_case}. Try to reach the goal.")
    # Close the case if reached the goal
    def close(self):
        if self.cases_layer == 0:
            print(f"--> #{len(self.context)+1} close(): There is no cases to close!"); return
        for fmla in self.context:
            if fmla.eq(self.goal):
                print(f"📁 Case closed! {self.current_case}'s goal reached.")
                self.pop()   # Revert to checkpoint
                if len(self.cases_stack[-1]) > 0:  # Move on to next case
                    next_case = self.cases_stack[-1].pop()
                    self.push()    # checkpoint
                    self.current_case = next_case
                    self.add(next_case); self.current_cases_nth += 1
                    print(f"{len(self.context)}.{self.current_cases_nth}. Assuming case {next_case}. Try to reach the goal.")
                    return
                # All cases have been proved
                self.cases_layer -= 1
                self.add(self.goal)
                print(f"All cases closed! {self.goal} have been added to context.")
                return
        print(f"--> #{len(self.context)+1} close(): Sorry, cannot reach goal {self.goal} from assumption {self.current_case}.")
        return
        

In [53]:
# Axiom T1: a on L, a on M, a on N, b on L, c on M, d on N, c d sameside L, b c sameside N ⊢ ¬(b d sameside M)
# // 線線線共點，各線有甲乙丙各點，乙丙同乙丙同甲邊，甲乙同丙邊，則甲丙不同乙邊
pc = Proof( [ onLine(a,L), onLine(a,M), onLine(a,N), onLine(b,L), onLine(c,M), onLine(d,N), sameside(c,d,L), sameside(b,c,N) ],
           Not(sameside(b,d,M)) )
# 1. Extract non-incidence facts from the sameside premises using S3
pc.have(Not(onLine(c, L)), rule='S3')
pc.have(Not(onLine(d, L)), rule='S3')
pc.have(Not(onLine(b, N)), rule='S3')
pc.have(Not(onLine(c, N)), rule='S3')
# 2. Establish point and line distinctness via structural contradictions (G1 / None)
pc.have(L != M, rule='G1')  # c is on M but not on L
pc.have(N != M, rule='G1')  # c is on M but not on N
pc.have(a != b )  # b is on L, but a is on N and b is not on N
pc.have(a != c )  # c is on M, but a is on L and c is not on L
pc.have(a != d )  # d is on N, but a is on L and d is not on L
# 3. Prove that b and d do not lie on line M (otherwise lines collapse)
pc.have(Not(onLine(b, M)), rule='G1')
pc.have(Not(onLine(d, M)), rule='G1')
# 4. First DA2 Application: Evaluate configuration relative to lines L and M
# Vertex: a, Line 1: L (with b), Line 2: M (with c), Test Point: d
pc.have( (Angle(b, a, c) == Angle(b, a, d) + Angle(d, a, c)) == And(sameside(b, d, M), sameside(c, d, L)), rule='DA2')
# 5. Second DA2 Application: Evaluate configuration relative to lines N and M
# Vertex: a, Line 1: N (with d), Line 2: M (with c), Test Point: b
pc.have((Angle(d, a, c) == Angle(d, a, b) + Angle(b, a, c)) == And(sameside(d, b, M), sameside(c, b, N)), rule='DA2')
# 6. Introduce strict positivity of non-degenerate angles to seal the contradiction
pc.have(Angle(b, a, d) > 0, rule='DA1')
# 7. Z3's arithmetic solver handles the rest: 
# If sameside(b, d, M) is True, the system reduces to x = y + z AND z = y + x, forcing y = 0.
# This directly clashes with Angle(b, a, d) > 0.
pc.have(Not(sameside(b, d, M)) )
pc.qed('T1')

1. onLine(a, L)
2. onLine(a, M)
3. onLine(a, N)
4. onLine(b, L)
5. onLine(c, M)
6. onLine(d, N)
7. sameside(c, d, L)
8. sameside(b, c, N)    / Not(sameside(b, d, M))
--------------------
9. Not(onLine(c, L))	[S3]
10. Not(onLine(d, L))	[S3]
11. Not(onLine(b, N))	[S3]
12. Not(onLine(c, N))	[S3]
13. L != M	[G1]
14. N != M	[G1]
15. a != b	[None]
16. a != c	[None]
17. a != d	[None]
18. Not(onLine(b, M))	[G1]
19. Not(onLine(d, M))	[G1]
20. (Angle(b, a, c) == Angle(b, a, d) + Angle(d, a, c)) ==
And(sameside(b, d, M), sameside(c, d, L))	[DA2]
21. (Angle(d, a, c) == Angle(d, a, b) + Angle(b, a, c)) ==
And(sameside(d, b, M), sameside(c, b, N))	[DA2]
22. Angle(b, a, d) > 0	[DA1]
23. Not(sameside(b, d, M))	[None]
🔥 Success! You proved Not(sameside(b, d, M)) is true.
💾 Registered T1 as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, d, L, M, N],
       Implies(And(onLine(a, L),
                   onLine(a, M),
                   onLine(a, N),
                   onLine(b, L),
              

True

In [54]:
# Betweenness line construction BL: Given bet(a,b,c), exists line L such that a b c on L.
pc = Proof( [bet(a,b,c)], And( onLine(a,L), onLine(b,L), onLine(c,L) ) )
pc.construct('LC1', L, [a,b])
pc.have( onLine(c,L), 'B2' )
pc.have(And( onLine(a,L), onLine(b,L), onLine(c,L) ))
pc.qed('BL')

1. bet(a, b, c)    / And(onLine(a, L), onLine(b, L), onLine(c, L))
--------------------
2. Constructed L where And(onLine(a, L), onLine(b, L)) 	[LC1]
3. onLine(c, L)	[B2]
4. And(onLine(a, L), onLine(b, L), onLine(c, L))	[None]
🔥 Success! You proved And(onLine(a, L), onLine(b, L), onLine(c, L)) is true.
💾 Added Construction Rule BL: [lambda a, b, c: bet(a, b, c), lambda L, a, b, c: And(onLine(a, L), onLine(b, L), onLine(c, L))] into CONSTRUCTION RULES.


True

In [55]:
# Collinearity construction CL: Given coll(a,b,c), exists line L such that a b c on L.
pc = Proof( [coll(a,b,c)], And( onLine(a,L), onLine(b,L), onLine(c,L) ) )
pc.have( Or( bet(a,b,c), bet(b,a,c), bet(a,c,b), And(a==b,b==c), And(a==b,a!=c), And(b==c,a!=b), And(c==a,a!=b) ), 'L0' )
pc.cases( Or( bet(a,b,c), bet(b,a,c), bet(a,c,b), And(a==b,b==c), And(a==b,a!=c), And(b==c,a!=b), And(c==a,a!=b) ) )
pc.construct('BL', L, [a,b,c])
pc.close()
pc.construct('BL', L, [b,a,c])
pc.have(And( onLine(a,L), onLine(b,L), onLine(c,L) ) )
pc.close()
pc.construct('BL', L, [a,c,b])
pc.have(And( onLine(a,L), onLine(b,L), onLine(c,L) ) )
pc.close()
pc.construct('CP1', e, [a])
pc.construct('LC1', L, [a,e])
pc.have(And( onLine(a,L), onLine(b,L), onLine(c,L) ) )
pc.close()
pc.construct('LC1', L, [a,c])
pc.have( And( onLine(a,L), onLine(b,L), onLine(c,L) ) )
pc.close()
pc.construct('LC1', L, [a,b])
pc.have( And( onLine(a,L), onLine(b,L), onLine(c,L) ) )
pc.close()
pc.construct('LC1', L, [b,c])
pc.have( And( onLine(a,L), onLine(b,L), onLine(c,L) ) )
pc.close()
pc.qed('CL')

1. coll(a, b, c)    / And(onLine(a, L), onLine(b, L), onLine(c, L))
--------------------
2. Or(bet(a, b, c),
   bet(b, a, c),
   bet(a, c, b),
   And(a == b, b == c),
   And(a == b, a != c),
   And(b == c, a != b),
   And(c == a, a != b))	[L0]
3.1. Assuming case bet(a, b, c). Try to reach the goal.
4. Constructed L where And(onLine(a, L), onLine(b, L), onLine(c, L)) 	[BL]
📁 Case closed! bet(a, b, c)'s goal reached.
3.2. Assuming case bet(b, a, c). Try to reach the goal.
4. Constructed L where And(onLine(b, L), onLine(a, L), onLine(c, L)) 	[BL]
5. And(onLine(a, L), onLine(b, L), onLine(c, L))	[None]
📁 Case closed! bet(b, a, c)'s goal reached.
3.3. Assuming case bet(a, c, b). Try to reach the goal.
4. Constructed L where And(onLine(a, L), onLine(c, L), onLine(b, L)) 	[BL]
5. And(onLine(a, L), onLine(b, L), onLine(c, L))	[None]
📁 Case closed! bet(a, c, b)'s goal reached.
3.4. Assuming case And(a == b, b == c). Try to reach the goal.
4. Constructed e where e != a 	[CP1]
5. Constructed L wh

True

In [56]:
# Lemma sameray_if_bet: If bet(a,b,c), then a b are on the same ray from c.
pc = Proof( [bet(a,b,c)], sameray(a,b,c) )
pc.have( Distinct(a,b,c), 'B1' )
pc.have( Not(bet(a,c,b)), 'B1' )
pc.construct('CL', L, [a,b,c])
pc.have( Or( bet(c,b,a), bet(c,a,b), bet(a,c,b) ), 'B6' )
pc.have( sameray(a,b,c) )
pc.qed('sameray_if_bet')

1. bet(a, b, c)    / And(a != c, b != c, Or(a == b, bet(c, a, b), bet(c, b, a)))
--------------------
2. Distinct(a, b, c)	[B1]
3. Not(bet(a, c, b))	[B1]
4. Constructed L where And(onLine(a, L), onLine(b, L), onLine(c, L)) 	[CL]
5. Or(bet(c, b, a), bet(c, a, b), bet(a, c, b))	[B6]
6. And(a != c, b != c, Or(a == b, bet(c, a, b), bet(c, b, a)))	[None]
🔥 Success! You proved And(a != c, b != c, Or(a == b, bet(c, a, b), bet(c, b, a))) is true.
💾 Registered sameray_if_bet as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(bet(a, b, c),
               And(a != c,
                   b != c,
                   Or(a == b, bet(c, a, b), bet(c, b, a)))))


True

In [198]:
# Lemma coll_if_sameray: If a b are on the same ray from o, then a b o are collinear.
pc = Proof([sameray(a,b,o)], coll(a,b,o))
pc.have( coll(a,b,o), 'L0' )
pc.qed('coll_if_sameray')

1. And(a != o, b != o, Or(a == b, bet(o, a, b), bet(o, b, a)))    / coll(a, b, o)
--------------------
2. coll(a, b, o)	[L0]
🔥 Success! You proved coll(a, b, o) is true.
💾 Registered coll_if_sameray as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, o],
       Implies(And(a != o,
                   b != o,
                   Or(a == b, bet(o, a, b), bet(o, b, a))),
               coll(a, b, o)))


True

In [58]:
# DA4 syntactic sugar eqang_if_sameray: Angles sharing the same two rays are equal.
pc =  Proof( [sameray(a,d,b), sameray(c,e,b)], Angle(a,b,c) == Angle(d,b,e) )
pc.have( coll(a,d,b), 'coll_if_sameray')
pc.have( coll(c,e,b), 'coll_if_sameray')
pc.construct('CL', L, [a,d,b])
pc.construct('CL', M, [c,e,b])
pc.have( Angle(a,b,c) == Angle(d,b,e), 'DA4' )
pc.qed('eqang_if_sameray')

1. And(a != b, d != b, Or(a == d, bet(b, a, d), bet(b, d, a)))
2. And(c != b, e != b, Or(c == e, bet(b, c, e), bet(b, e, c)))    / Angle(a, b, c) == Angle(d, b, e)
--------------------
3. coll(a, d, b)	[coll_if_sameray]
4. coll(c, e, b)	[coll_if_sameray]
5. Constructed L where And(onLine(a, L), onLine(d, L), onLine(b, L)) 	[CL]
6. Constructed M where And(onLine(c, M), onLine(e, M), onLine(b, M)) 	[CL]
7. Angle(a, b, c) == Angle(d, b, e)	[DA4]
🔥 Success! You proved Angle(a, b, c) == Angle(d, b, e) is true.
💾 Registered eqang_if_sameray as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, d, e],
       Implies(And(And(a != b,
                       d != b,
                       Or(a == d,
                          bet(b, a, d),
                          bet(b, d, a))),
                   And(c != b,
                       e != b,
                       Or(c == e,
                          bet(b, c, e),
                          bet(b, e, c)))),
               Angle(a, b, c) == An

True

In [59]:
## Elements Proposition 1: Given distinct points a b, construct point c such that ab = bc = ca.  (construct equil triangle)
pc = Proof([a!=b ], And(Segment(a,b)==Segment(b,c), Segment(b,c)==Segment(c,a)) ) 
pc.construct('LC2', α,[a,b])
pc.construct('LC2', β, [b,a])
pc.have( And(inside(a,α), inside(b,β)), 'G3' )
pc.have(interCC(α,β), 'I5')
pc.construct('CI6', c, [α,β])
pc.have(Segment(a,c) == Segment(a,b), 'DS3')
pc.have(Segment(b,c) == Segment(b,a), 'DS3')
pc.have(And(Segment(a,b)==Segment(b,c), Segment(b,c)==Segment(c,a)))
pc.qed('E1')

1. a != b    / And(Segment(a, b) == Segment(b, c),
    Segment(b, c) == Segment(c, a))
--------------------
2. Constructed α where And(center(a, α), onCircle(b, α)) 	[LC2]
3. Constructed β where And(center(b, β), onCircle(a, β)) 	[LC2]
4. And(inside(a, α), inside(b, β))	[G3]
5. interCC(α, β)	[I5]
6. Constructed c where And(onCircle(c, α), onCircle(c, β)) 	[CI6]
7. Segment(a, c) == Segment(a, b)	[DS3]
8. Segment(b, c) == Segment(b, a)	[DS3]
9. And(Segment(a, b) == Segment(b, c),
    Segment(b, c) == Segment(c, a))	[None]
🔥 Success! You proved And(Segment(a, b) == Segment(b, c),
    Segment(b, c) == Segment(c, a)) is true.
💾 Added Construction Rule E1: [lambda a, b: a != b, lambda c, a, b: And(Segment(a, b) == Segment(b, c),
    Segment(b, c) == Segment(c, a))] into CONSTRUCTION RULES.


True

In [60]:
## Prop1's Corollary: If ab = bc = ca then Distinct(a,b,c).
pc = Proof([ a!=b ], And(Segment(a,b)==Segment(b,c), Segment(b,c)==Segment(c,a), Distinct(a,b,c)) ) 
pc.construct('E1', c, [a,b])
pc.have(Segment(a,b) != 0, 'M1')
pc.have(Segment(b,c) != 0)
pc.have( b != c, 'M1')
pc.have(Segment(c,a) != 0)
pc.have( c != a, 'M1')
pc.have( Distinct(a,b,c))
pc.have( And(Segment(a,b)==Segment(b,c), Segment(b,c)==Segment(c,a), Distinct(a,b,c)) )
pc.qed('E1a')

1. a != b    / And(Segment(a, b) == Segment(b, c),
    Segment(b, c) == Segment(c, a),
    Distinct(a, b, c))
--------------------
2. Constructed c where And(Segment(a, b) == Segment(b, c),
    Segment(b, c) == Segment(c, a)) 	[E1]
3. Segment(a, b) != 0	[M1]
4. Segment(b, c) != 0	[None]
5. b != c	[M1]
6. Segment(c, a) != 0	[None]
7. c != a	[M1]
8. Distinct(a, b, c)	[None]
9. And(Segment(a, b) == Segment(b, c),
    Segment(b, c) == Segment(c, a),
    Distinct(a, b, c))	[None]
🔥 Success! You proved And(Segment(a, b) == Segment(b, c),
    Segment(b, c) == Segment(c, a),
    Distinct(a, b, c)) is true.
💾 Added Construction Rule E1a: [lambda a, b: a != b, lambda c, a, b: And(Segment(a, b) == Segment(b, c),
    Segment(b, c) == Segment(c, a),
    Distinct(a, b, c))] into CONSTRUCTION RULES.


True

In [61]:
# Proposition 2: Given distinct points a b c, construct point f such that af = bc.   (non-collapsing compass)
pc = Proof( [Distinct(a,b,c)], Segment(a,f) == Segment(b,c) )
pc.construct( 'E1a', d, [a,b] )
pc.construct('LC2', β, [b,c])
pc.have(inside(b,β), 'G3')
pc.construct('LC1', M, [d,a])
pc.construct('LC1', N, [d,b])
pc.have( interLC(N, β), 'I3')
pc.construct('CI5', g, [b,d,N,β])
pc.have( Segment(d,g) == Segment(d,b)+Segment(b,g), 'DS1' )
pc.have( Segment(d,g) == Segment(d,a)+Segment(b,g))
pc.have( Segment(b,g) >= 0, 'M2')
pc.have( Segment(b,g) != 0, 'M1')
pc.have( Segment(b,g) > 0)
pc.have( Segment(d,a) < Segment(d,g) )
pc.construct('LC2', α, [d,g])
pc.have(inside(a,α), 'DS4')    # [since d is the center and da < dg].
pc.have( d != a )
pc.construct('CI5', f, [a,d,M,α])
pc.have( Segment(d,f) == Segment(d,a) + Segment(a,f), 'DS1' )
pc.have( Segment(d,f) == Segment(d,g) , 'DS3' )
pc.have( Segment(d,a) + Segment(a,f) == Segment(d,a) + Segment(b,g) )
pc.have( Segment(a,f) == Segment(b,g) )
pc.have( Segment(b,g) == Segment(b,c), 'DS3' )
pc.have( Segment(a,f) == Segment(b,c) )
pc.qed('E2')

1. Distinct(a, b, c)    / Segment(a, f) == Segment(b, c)
--------------------
2. Constructed d where And(Segment(a, b) == Segment(b, d),
    Segment(b, d) == Segment(d, a),
    Distinct(a, b, d)) 	[E1a]
3. Constructed β where And(center(b, β), onCircle(c, β)) 	[LC2]
4. inside(b, β)	[G3]
5. Constructed M where And(onLine(d, M), onLine(a, M)) 	[LC1]
6. Constructed N where And(onLine(d, N), onLine(b, N)) 	[LC1]
7. interLC(N, β)	[I3]
8. Constructed g where And(onCircle(g, β), onLine(g, N), bet(g, b, d)) 	[CI5]
9. Segment(d, g) == Segment(d, b) + Segment(b, g)	[DS1]
10. Segment(d, g) == Segment(d, a) + Segment(b, g)	[None]
11. Segment(b, g) >= 0	[M2]
12. Segment(b, g) != 0	[M1]
13. Segment(b, g) > 0	[None]
14. Segment(d, a) < Segment(d, g)	[None]
15. Constructed α where And(center(d, α), onCircle(g, α)) 	[LC2]
16. inside(a, α)	[DS4]
17. d != a	[None]
18. Constructed f where And(onCircle(f, α), onLine(f, M), bet(f, a, d)) 	[CI5]
19. Segment(d, f) == Segment(d, a) + Segment(a, f)	[DS1]
20. Se

True

In [62]:
# Aux 2a
pc = Proof( [a==b, b==c], Segment(a,f) == Segment(b,c) )
pc.have( Segment(a,a) == 0 )
pc.have( Segment(a,b) == 0 )
pc.have( Segment(b,c) == 0 )
pc.construct('CP0', f, [a])
pc.have( Segment(a,f) == 0 )
pc.have( Segment(a,f) == Segment(b,c) )
pc.qed('A2a')

1. a == b
2. b == c    / Segment(a, f) == Segment(b, c)
--------------------
3. Segment(a, a) == 0	[None]
4. Segment(a, b) == 0	[None]
5. Segment(b, c) == 0	[None]
6. Constructed f where f == a 	[CP0]
7. Segment(a, f) == 0	[None]
8. Segment(a, f) == Segment(b, c)	[None]
🔥 Success! You proved Segment(a, f) == Segment(b, c) is true.
💾 Added Construction Rule A2a: [lambda a, b, c: And(a == b, b == c), lambda f, a, b, c: Segment(a, f) == Segment(b, c)] into CONSTRUCTION RULES.


True

In [63]:
# Aux 2b
pc = Proof( [a==b, b!=c], Segment(a,f) == Segment(b,c) )
pc.construct('CP0', f, [c])
pc.have(Segment(a,f) == Segment(a,c) )
pc.have(Segment(a,f) == Segment(b,c) )
pc.qed('A2b')

1. a == b
2. b != c    / Segment(a, f) == Segment(b, c)
--------------------
3. Constructed f where f == c 	[CP0]
4. Segment(a, f) == Segment(a, c)	[None]
5. Segment(a, f) == Segment(b, c)	[None]
🔥 Success! You proved Segment(a, f) == Segment(b, c) is true.
💾 Added Construction Rule A2b: [lambda a, b, c: And(a == b, b != c), lambda f, a, b, c: Segment(a, f) == Segment(b, c)] into CONSTRUCTION RULES.


True

In [64]:
# Aux 2c
pc = Proof( [a!=b, b==c], Segment(a,f) == Segment(b,c) )
pc.construct('CP0', f, [a])
pc.have(Segment(b,c) == 0 )
pc.have(Segment(a,f) == 0 )
pc.have(Segment(a,f) == Segment(b,c))
pc.qed('A2c')

1. a != b
2. b == c    / Segment(a, f) == Segment(b, c)
--------------------
3. Constructed f where f == a 	[CP0]
4. Segment(b, c) == 0	[None]
5. Segment(a, f) == 0	[None]
6. Segment(a, f) == Segment(b, c)	[None]
🔥 Success! You proved Segment(a, f) == Segment(b, c) is true.
💾 Added Construction Rule A2c: [lambda a, b, c: And(a != b, b == c), lambda f, a, b, c: Segment(a, f) == Segment(b, c)] into CONSTRUCTION RULES.


True

In [65]:
# Aux 2d
pc = Proof( [a!=b, a==c], Segment(a,f) == Segment(b,c) )
pc.construct('CP0', f, [b])
pc.have(Segment(a,f) == Segment(b,a))
pc.have(Segment(a,f) == Segment(b,c))
pc.qed('A2d')

1. a != b
2. a == c    / Segment(a, f) == Segment(b, c)
--------------------
3. Constructed f where f == b 	[CP0]
4. Segment(a, f) == Segment(b, a)	[None]
5. Segment(a, f) == Segment(b, c)	[None]
🔥 Success! You proved Segment(a, f) == Segment(b, c) is true.
💾 Added Construction Rule A2d: [lambda a, b, c: And(a != b, a == c), lambda f, a, b, c: Segment(a, f) == Segment(b, c)] into CONSTRUCTION RULES.


True

In [66]:
# Prop 2a: Given any points a b c, construct f such that af = bc   (non-collapsing compass, no distinct requirement)
pc = Proof( [points(a,b,c)], Segment(a,f) == Segment(b,c) )
pc.have(Or(
    And(a==b, b==c),  # 1. All three are the same point
    And(a==b, b!=c),  # 2. a and b are same, c is different
    And(a!=b, b==c),  # 3. b and c are same, a is different
    And(a==c, a!=b),  # 4. a and c are same, b is different (The missing piece!)
    Distinct(a,b,c)   # 5. All three are completely separate points
))
pc.cases(Or(And(a==b, b==c), And(a==b, b!=c), And(a!=b, b==c), And(a==c, a!=b), Distinct(a,b,c) ))
pc.construct('A2a', f, [])
pc.close()
pc.construct('A2b', f, [])
pc.close()
pc.construct('A2c', f, [])
pc.close()
pc.construct('A2d', f, [])
pc.close()
pc.construct('E2', f, [])
pc.close()
pc.qed('E2a')

1. Or(Distinct(a, b, c), Not(Distinct(a, b, c)))    / Segment(a, f) == Segment(b, c)
--------------------
2. Or(And(a == b, b == c),
   And(a == b, b != c),
   And(a != b, b == c),
   And(a == c, a != b),
   Distinct(a, b, c))	[None]
3.1. Assuming case And(a == b, b == c). Try to reach the goal.
4. Constructed f where Segment(a, f) == Segment(b, c) 	[A2a]
📁 Case closed! And(a == b, b == c)'s goal reached.
3.2. Assuming case And(a == b, b != c). Try to reach the goal.
4. Constructed f where Segment(a, f) == Segment(b, c) 	[A2b]
📁 Case closed! And(a == b, b != c)'s goal reached.
3.3. Assuming case And(a != b, b == c). Try to reach the goal.
4. Constructed f where Segment(a, f) == Segment(b, c) 	[A2c]
📁 Case closed! And(a != b, b == c)'s goal reached.
3.4. Assuming case And(a == c, a != b). Try to reach the goal.
4. Constructed f where Segment(a, f) == Segment(b, c) 	[A2d]
📁 Case closed! And(a == c, a != b)'s goal reached.
3.5. Assuming case Distinct(a, b, c). Try to reach the goal.
4. Co

True

In [67]:
# Proposition 3: Given segments ab > cd, construct e on ab such that ae = cd   (cut out a shorter segment from longer segment)
pc = Proof([  a!=b, c!=d, Segment(a,b)>Segment(c,d)], 
           And(bet(a,e,b), Segment(a,e)==Segment(c,d))  )
pc.have( Segment(a,b) > 0, 'M1' )
pc.have( Segment(c,d)>0, 'M1' )
pc.construct('E2a', f, [a,c,d])
pc.have( Segment(a,f) < Segment(a,b))
pc.construct('LC2', α, [a,f])
pc.have( inside(a,α), 'G3' )
pc.have( outside(b,α), 'DS4' )
pc.construct('LC1', L, [a,b])
pc.construct('CI4', e, [a,b,L,α])
pc.have( Segment(a,e) == Segment(a,f), 'DS3' )
pc.have( Segment(a, e) == Segment(c, d) )
pc.have( And(bet(a, e, b), Segment(a, e) == Segment(c, d)) )
pc.qed('E3')

1. a != b
2. c != d
3. Segment(a, b) > Segment(c, d)    / And(bet(a, e, b), Segment(a, e) == Segment(c, d))
--------------------
4. Segment(a, b) > 0	[M1]
5. Segment(c, d) > 0	[M1]
6. Constructed f where Segment(a, f) == Segment(c, d) 	[E2a]
7. Segment(a, f) < Segment(a, b)	[None]
8. Constructed α where And(center(a, α), onCircle(f, α)) 	[LC2]
9. inside(a, α)	[G3]
10. And(Not(inside(b, α)), Not(onCircle(b, α)))	[DS4]
11. Constructed L where And(onLine(a, L), onLine(b, L)) 	[LC1]
12. Constructed e where And(onCircle(e, α), onLine(e, L), bet(a, e, b)) 	[CI4]
13. Segment(a, e) == Segment(a, f)	[DS3]
14. Segment(a, e) == Segment(c, d)	[None]
15. And(bet(a, e, b), Segment(a, e) == Segment(c, d))	[None]
🔥 Success! You proved And(bet(a, e, b), Segment(a, e) == Segment(c, d)) is true.
💾 Added Construction Rule E3: [lambda a, b, c, d: And(a != b, c != d, Segment(a, b) > Segment(c, d)), lambda e, a, b, c, d: And(bet(a, e, b), Segment(a, e) == Segment(c, d))] into CONSTRUCTION RULES.


True

In [127]:
# Lemma sameray_if_sameside_coll: If c on L, a b sameside L, a b c collinear, then a b sameray c.
pc = Proof( [onLine(c,L), sameside(a,b,L), coll(a,b,c)], sameray(a,b,c) )
pc.have( And(a!=c, b!=c), 'S3' )
pc.have( Not(bet(a,c,b)), 'P3' )
pc.have( sameray(a,b,c), 'L0' )
pc.qed('sameray_if_sameside_coll')

1. onLine(c, L)
2. sameside(a, b, L)
3. coll(a, b, c)    / And(a != c, b != c, Or(a == b, bet(c, a, b), bet(c, b, a)))
--------------------
4. And(a != c, b != c)	[S3]
5. Not(bet(a, c, b))	[P3]
6. And(a != c, b != c, Or(a == b, bet(c, a, b), bet(c, b, a)))	[L0]
🔥 Success! You proved And(a != c, b != c, Or(a == b, bet(c, a, b), bet(c, b, a))) is true.
💾 Registered sameray_if_sameside_coll as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, L],
       Implies(And(onLine(c, L),
                   sameside(a, b, L),
                   coll(a, b, c)),
               And(a != c,
                   b != c,
                   Or(a == b, bet(c, a, b), bet(c, b, a)))))


True

In [128]:
# Lemma ne_if_int: If p inside angle abc, then Distinct(ab, bc, ac) and Distinct(a,b,c).
pc = Proof( [interior(p,a,b,c)], And(Distinct(ab, bc, ac), Distinct(a,b,c)) )
pc.have( Not(onLine(a, bc)), 'S3' )
pc.have( Not(onLine(c, ab)), 'S3' )
pc.have( ab != bc ); pc.have( ac != bc )
pc.have( And(Distinct(ab, bc, ac), Distinct(a,b,c)) )
pc.qed('ne_if_int')

1. And(sameside(p, a, line(b, c)), sameside(p, c, line(a, b)))    / And(Distinct(line(a, b), line(b, c), line(a, c)),
    Distinct(a, b, c))
--------------------
2. Not(onLine(a, line(b, c)))	[S3]
3. Not(onLine(c, line(a, b)))	[S3]
4. line(a, b) != line(b, c)	[None]
5. line(a, c) != line(b, c)	[None]
6. And(Distinct(line(a, b), line(b, c), line(a, c)),
    Distinct(a, b, c))	[None]
🔥 Success! You proved And(Distinct(line(a, b), line(b, c), line(a, c)),
    Distinct(a, b, c)) is true.
💾 Registered ne_if_int as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, p],
       Implies(And(sameside(p, a, line(b, c)),
                   sameside(p, c, line(a, b))),
               And(Distinct(line(a, b),
                            line(b, c),
                            line(a, c)),
                   Distinct(a, b, c))))


True

In [129]:
# Lemma non_if_tri: The vertices of a triangle are not on their opposite sides
pc = Proof( [triangle(a,b,c)], And( Not(onLine(a,bc)), Not(onLine(b,ac)), Not(onLine(c,ab)) ) )
pc.have( Not( And( onLine(a,ab), onLine(b,ab), onLine(c,ab) ) ), 'L4' )
pc.have( Not(onLine(c,ab)) )
pc.have( Not( And( onLine(a,ac), onLine(b,ac), onLine(c,ac) ) ), 'L4' )
pc.have( Not(onLine(b,ac)) )
pc.have( Not( And( onLine(a,bc), onLine(b,bc), onLine(c,bc) ) ), 'L4' )
pc.have( Not(onLine(a,bc)) )
pc.have(And( Not(onLine(a,bc)), Not(onLine(b,ac)), Not(onLine(c,ab)) )  )
pc.qed('non_if_tri')

1. Not(coll(a, b, c))    / And(Not(onLine(a, line(b, c))),
    Not(onLine(b, line(a, c))),
    Not(onLine(c, line(a, b))))
--------------------
2. Not(And(onLine(a, line(a, b)),
        onLine(b, line(a, b)),
        onLine(c, line(a, b))))	[L4]
3. Not(onLine(c, line(a, b)))	[None]
4. Not(And(onLine(a, line(a, c)),
        onLine(b, line(a, c)),
        onLine(c, line(a, c))))	[L4]
5. Not(onLine(b, line(a, c)))	[None]
6. Not(And(onLine(a, line(b, c)),
        onLine(b, line(b, c)),
        onLine(c, line(b, c))))	[L4]
7. Not(onLine(a, line(b, c)))	[None]
8. And(Not(onLine(a, line(b, c))),
    Not(onLine(b, line(a, c))),
    Not(onLine(c, line(a, b))))	[None]
🔥 Success! You proved And(Not(onLine(a, line(b, c))),
    Not(onLine(b, line(a, c))),
    Not(onLine(c, line(a, b)))) is true.
💾 Registered non_if_tri as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(Not(coll(a, b, c)),
               And(Not(onLine(a, line(b, c))),
                   Not(onLine(b, line(a

True

In [144]:
# Lemma on_if_bet: If bet(a,b,c), then a on bc, b on ac, c on ab.
pc = Proof( [bet(a,b,c)], And( onLine(a,bc), onLine(b,ac), onLine(c,ab) ) )
pc.construct('LC1', L, [a,b])
pc.have( onLine(a,bc), 'B2' )
pc.have( onLine(b,ac), 'B3' )
pc.have( And( onLine(a,bc), onLine(b,ac), onLine(c,ab) ), 'B2' )
pc.qed('on_if_bet')

1. bet(a, b, c)    / And(onLine(a, line(b, c)),
    onLine(b, line(a, c)),
    onLine(c, line(a, b)))
--------------------
2. Constructed L where And(onLine(a, L), onLine(b, L)) 	[LC1]
3. onLine(a, line(b, c))	[B2]
4. onLine(b, line(a, c))	[B3]
5. And(onLine(a, line(b, c)),
    onLine(b, line(a, c)),
    onLine(c, line(a, b)))	[B2]
🔥 Success! You proved And(onLine(a, line(b, c)),
    onLine(b, line(a, c)),
    onLine(c, line(a, b))) is true.
💾 Registered on_if_bet as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(bet(a, b, c),
               And(onLine(a, line(b, c)),
                   onLine(b, line(a, c)),
                   onLine(c, line(a, b)))))


True

In [187]:
# Lemma eq_if_coll: If coll(a,b,c), then lines ab = ac = bc.
pc = Proof( [coll(a,b,c), a!=b, a!=c ], ab == ac )
pc.construct('CL', L, [a, b,c] )
pc.have( ab == L , 'G1' )
pc.have( ab == ac , 'G1' )
pc.qed('eq_if_coll')

1. coll(a, b, c)
2. a != b
3. a != c    / line(a, b) == line(a, c)
--------------------
4. Constructed L where And(onLine(a, L), onLine(b, L), onLine(c, L)) 	[CL]
5. line(a, b) == L	[G1]
6. line(a, b) == line(a, c)	[G1]
🔥 Success! You proved line(a, b) == line(a, c) is true.
💾 Registered eq_if_coll as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(And(coll(a, b, c), a != b, a != c),
               line(a, b) == line(a, c)))


True

In [188]:
# Lemma on_if_coll:
pc = Proof( [Distinct(a,b,c), coll(a,b,c)],  And( onLine(a,bc), onLine(b,ac), onLine(c,ab) ) )
pc.construct('CL', L, [a,b,c])
pc.have( Or( bet(a,b,c), bet(b,c,a), bet(c,a,b), a==b, b==c, a==c ) )
pc.have( And(ac==L,bc==L,ac==L), 'G1' )
pc.have( And( onLine(a,bc), onLine(b,ac), onLine(c,ab)  ), 'G1' )
pc.qed('on_if_coll')

1. Distinct(a, b, c)
2. coll(a, b, c)    / And(onLine(a, line(b, c)),
    onLine(b, line(a, c)),
    onLine(c, line(a, b)))
--------------------
3. Constructed L where And(onLine(a, L), onLine(b, L), onLine(c, L)) 	[CL]
4. Or(bet(a, b, c),
   bet(b, c, a),
   bet(c, a, b),
   a == b,
   b == c,
   a == c)	[None]
5. And(line(a, c) == L, line(b, c) == L, line(a, c) == L)	[G1]
6. And(onLine(a, line(b, c)),
    onLine(b, line(a, c)),
    onLine(c, line(a, b)))	[G1]
🔥 Success! You proved And(onLine(a, line(b, c)),
    onLine(b, line(a, c)),
    onLine(c, line(a, b))) is true.
💾 Registered on_if_coll as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(And(Distinct(a, b, c), coll(a, b, c)),
               And(onLine(a, line(b, c)),
                   onLine(b, line(a, c)),
                   onLine(c, line(a, b)))))


True

In [189]:
# Lemma tri_if_non: If b!=c, a not on bc, then abc forms a triangle.
pc = Proof( [ b!=c, Not(onLine(a,bc))], triangle(a,b,c) )
pc.construct('LC1', L, [b,c])
pc.have( L==bc, 'G1' )
pc.have( Distinct(a,b,c) )
pc.have( Implies(coll(a,b,c), Or( bet(a,b,c),bet(b,a,c),bet(a,c,b) ) ) )
pc.have( Implies( coll(a,b,c), onLine(a,L) ), 'on_if_bet' )
pc.have( triangle(a,b,c) )
pc.qed('tri_if_non')

1. b != c
2. Not(onLine(a, line(b, c)))    / Not(coll(a, b, c))
--------------------
3. Constructed L where And(onLine(b, L), onLine(c, L)) 	[LC1]
4. L == line(b, c)	[G1]
5. Distinct(a, b, c)	[None]
6. Implies(coll(a, b, c),
        Or(bet(a, b, c), bet(b, a, c), bet(a, c, b)))	[None]
7. Implies(coll(a, b, c), onLine(a, L))	[on_if_bet]
8. Not(coll(a, b, c))	[None]
🔥 Success! You proved Not(coll(a, b, c)) is true.
💾 Registered tri_if_non as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(And(b != c, Not(onLine(a, line(b, c)))),
               Not(coll(a, b, c))))


True

In [190]:
# Lemma tri_if_int: If p inside angle abc, then abc forms a triangle.
pc = Proof( [interior(p,a,b,c)], triangle(a,b,c) )
pc.have( Not(onLine(a, bc)), 'S3' )
pc.have( Not(onLine(c, ab)), 'S3' )
pc.have( Distinct(a,b,c) )
pc.have( triangle(a,b,c), 'tri_if_non' )
pc.qed('tri_if_int')

1. And(sameside(p, a, line(b, c)), sameside(p, c, line(a, b)))    / Not(coll(a, b, c))
--------------------
2. Not(onLine(a, line(b, c)))	[S3]
3. Not(onLine(c, line(a, b)))	[S3]
4. Distinct(a, b, c)	[None]
5. Not(coll(a, b, c))	[tri_if_non]
🔥 Success! You proved Not(coll(a, b, c)) is true.
💾 Registered tri_if_int as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, p],
       Implies(And(sameside(p, a, line(b, c)),
                   sameside(p, c, line(a, b))),
               Not(coll(a, b, c))))


True

In [191]:
# Lemma ne_if_diff If a b diffside L, then a != b
pc = Proof( [diffside(a,b,L) ], a != b )
pc.have( Implies( a==b,  Not(sameside(a, a, L))  ))
pc.have( a!=b, 'S1')
pc.qed('ne_if_diff')

1. And(Not(onLine(a, L)),
    Not(onLine(b, L)),
    Not(sameside(a, b, L)))    / a != b
--------------------
2. Implies(a == b, Not(sameside(a, a, L)))	[None]
3. a != b	[S1]
🔥 Success! You proved a != b is true.
💾 Registered ne_if_diff as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, L],
       Implies(And(Not(onLine(a, L)),
                   Not(onLine(b, L)),
                   Not(sameside(a, b, L))),
               a != b))


True

In [192]:
# Lemma eq_if_bet: If bet(a,b,c), then lines ab = ac = bc.
pc = Proof( [bet(a,b,c)], And( ab == ac, ab == bc, ac == bc ) )
pc.construct('LC1', L, [a, b] )
pc.have( onLine(c,L), 'B2' )
pc.have( And( ab == ac, ab == bc, ac == bc ), 'G1' )
pc.qed('eq_if_bet')

1. bet(a, b, c)    / And(line(a, b) == line(a, c),
    line(a, b) == line(b, c),
    line(a, c) == line(b, c))
--------------------
2. Constructed L where And(onLine(a, L), onLine(b, L)) 	[LC1]
3. onLine(c, L)	[B2]
4. And(line(a, b) == line(a, c),
    line(a, b) == line(b, c),
    line(a, c) == line(b, c))	[G1]
🔥 Success! You proved And(line(a, b) == line(a, c),
    line(a, b) == line(b, c),
    line(a, c) == line(b, c)) is true.
💾 Registered eq_if_bet as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(bet(a, b, c),
               And(line(a, b) == line(a, c),
                   line(a, b) == line(b, c),
                   line(a, c) == line(b, c))))


True

In [193]:
# Lemma tri_if_non_ne: If b!=c and a not on bc, then abc forms a triangle.
pc = Proof( [b!=c, Not(onLine(a,bc))], triangle(a,b,c) )
pc.have( Implies( coll(a,b,c), onLine(a,bc) ), 'eq_if_coll')
pc.have(triangle(a,b,c) )
pc.qed('tri_if_non_ne')

1. b != c
2. Not(onLine(a, line(b, c)))    / Not(coll(a, b, c))
--------------------
3. Implies(coll(a, b, c), onLine(a, line(b, c)))	[eq_if_coll]
4. Not(coll(a, b, c))	[None]
🔥 Success! You proved Not(coll(a, b, c)) is true.
💾 Registered tri_if_non_ne as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(And(b != c, Not(onLine(a, line(b, c)))),
               Not(coll(a, b, c))))


True

In [196]:
# Lemma ne_if_tri: If a b c forms triangle, then the three sides are different.
pc = Proof( [triangle(a,b,c)], Distinct(ab,ac,bc) )
pc.have( Not(onLine(c,ab)), 'non_if_tri' )
pc.have( ab != ac, 'G1' )
pc.have( Not(onLine(a,bc)), 'non_if_tri' )
pc.have( Distinct(ab,ac,bc), 'G1' )
pc.qed('ne_if_tri')

1. Not(coll(a, b, c))    / Distinct(line(a, b), line(a, c), line(b, c))
--------------------
2. Not(onLine(c, line(a, b)))	[non_if_tri]
3. line(a, b) != line(a, c)	[G1]
4. Not(onLine(a, line(b, c)))	[non_if_tri]
5. Distinct(line(a, b), line(a, c), line(b, c))	[G1]
🔥 Success! You proved Distinct(line(a, b), line(a, c), line(b, c)) is true.
💾 Registered ne_if_tri as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(Not(coll(a, b, c)),
               Distinct(line(a, b), line(a, c), line(b, c))))


True

In [199]:
# Lemma R2: If b c sameray a, a on L, b ~on L, then b c sameside L
pc = Proof( [ sameray(b,c,a), onLine(a,L), Not(onLine(b,L)) ], sameside(b,c,L) )
pc.have( Or(b == c, bet(a, b, c), bet(a, c, b) ))
pc.cases( Or(b == c, bet(a, b, c), bet(a, c, b) ))
pc.have( sameside(b,c,L), 'S1' )
pc.close()
pc.have( sameside(b,c,L), 'P2' )
pc.close()
pc.have(coll(b,c,a), 'coll_if_sameray')
pc.construct('CL', M, [b,c,a])
pc.have( M != L, 'S3' )
pc.have( Not(onLine(c,L)), 'G1' )
pc.have( sameside(b,c,L), 'P2' )
pc.close()
pc.qed('R2')

1. And(b != a, c != a, Or(b == c, bet(a, b, c), bet(a, c, b)))
2. onLine(a, L)
3. Not(onLine(b, L))    / sameside(b, c, L)
--------------------
4. Or(b == c, bet(a, b, c), bet(a, c, b))	[None]
5.1. Assuming case b == c. Try to reach the goal.
6. sameside(b, c, L)	[S1]
📁 Case closed! b == c's goal reached.
5.2. Assuming case bet(a, b, c). Try to reach the goal.
6. sameside(b, c, L)	[P2]
📁 Case closed! bet(a, b, c)'s goal reached.
5.3. Assuming case bet(a, c, b). Try to reach the goal.
6. coll(b, c, a)	[coll_if_sameray]
7. Constructed M where And(onLine(b, M), onLine(c, M), onLine(a, M)) 	[CL]
8. M != L	[S3]
9. Not(onLine(c, L))	[G1]
10. sameside(b, c, L)	[P2]
📁 Case closed! bet(a, c, b)'s goal reached.
All cases closed! sameside(b, c, L) have been added to context.
🔥 Success! You proved sameside(b, c, L) is true.
💾 Registered R2 as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, L],
       Implies(And(And(b != a,
                       c != a,
                       Or(b == c,


True

In [203]:
# Lemma bet_from_diff: If a b diffside L, construct e on L such that bet(a,e,b)
pc = Proof( [diffside(a,b,L)], And( onLine(e,L), bet(a,e,b) ) )
pc.have( interLL(L,ab), 'I1' )
pc.construct('CI1', e, [L,ab])
pc.have( L != ab )
pc.have( a != b, 'ne_if_diff' )
pc.have( Distinct(a,e,b) )
pc.have(  And( onLine(e,L), bet(a,e,b) ), 'P4' )
pc.qed('bet_from_diff')

1. And(Not(onLine(a, L)),
    Not(onLine(b, L)),
    Not(sameside(a, b, L)))    / And(onLine(e, L), bet(a, e, b))
--------------------
2. interLL(L, line(a, b))	[I1]
3. Constructed e where And(onLine(e, L), onLine(e, line(a, b))) 	[CI1]
4. L != line(a, b)	[None]
5. a != b	[ne_if_diff]
6. Distinct(a, e, b)	[None]
7. And(onLine(e, L), bet(a, e, b))	[P4]
🔥 Success! You proved And(onLine(e, L), bet(a, e, b)) is true.
💾 Added Construction Rule bet_from_diff: [lambda a, b, L: And(Not(onLine(a, L)),
    Not(onLine(b, L)),
    Not(sameside(a, b, L))), lambda e, a, b, L: And(onLine(e, L), bet(a, e, b))] into CONSTRUCTION RULES.


True

In [204]:
# Lemma sameside_if_samediff: If abc forms triangle, c d sameside ab, b d diffside ac, then b c sameside ad.
pc = Proof( [triangle(a,b,c), sameside(c,d,ab), diffside(b,d,ac) ], sameside(b,c,ad) )
pc.construct('bet_from_diff', e, [b,d,ac])
pc.have( coll(a,c,e), 'L4' )
pc.have( be == bd, 'eq_if_bet' )
pc.have( Not(onLine(d,ab)), 'S3' )
pc.have( triangle(a,b,d), 'tri_if_non_ne' )
pc.have( ab != bd, 'ne_if_tri' )
pc.have( bd == be, 'eq_if_bet' )
pc.have( a!=e, 'G1' )
pc.have( ae == ac, 'eq_if_coll' )
pc.have( Not(onLine(e,ad)), 'G1' )
pc.have( sameside(b,e,ad), 'P2' )
# Trying to get sameray(c,e,a)
pc.have( Not(onLine(e,ab)), 'G1' )
pc.have( sameside(d,e,ab), 'P2' )
pc.have( sameside(c,e,ab), 'S4' )
pc.have( Implies( bet(c,a,e), diffside(c,e,ab) ), 'P3' )
pc.have( Not(bet(c,a,e)) )
pc.have( sameray(c,e,a) )
pc.have( sameside(e,c,ad), 'R2' )
pc.have( sameside(b,c,ad), 'S4' )
pc.qed('sameside_if_samediff')

1. Not(coll(a, b, c))
2. sameside(c, d, line(a, b))
3. And(Not(onLine(b, line(a, c))),
    Not(onLine(d, line(a, c))),
    Not(sameside(b, d, line(a, c))))    / sameside(b, c, line(a, d))
--------------------
4. Constructed e where And(onLine(e, line(a, c)), bet(b, e, d)) 	[bet_from_diff]
5. coll(a, c, e)	[L4]
6. line(b, e) == line(b, d)	[eq_if_bet]
7. Not(onLine(d, line(a, b)))	[S3]
8. Not(coll(a, b, d))	[tri_if_non_ne]
9. line(a, b) != line(b, d)	[ne_if_tri]
10. line(b, d) == line(b, e)	[eq_if_bet]
11. a != e	[G1]
12. line(a, e) == line(a, c)	[eq_if_coll]
13. Not(onLine(e, line(a, d)))	[G1]
14. sameside(b, e, line(a, d))	[P2]
15. Not(onLine(e, line(a, b)))	[G1]
16. sameside(d, e, line(a, b))	[P2]
17. sameside(c, e, line(a, b))	[S4]
18. Implies(bet(c, a, e),
        And(Not(onLine(c, line(a, b))),
            Not(onLine(e, line(a, b))),
            Not(sameside(c, e, line(a, b)))))	[P3]
19. Not(bet(c, a, e))	[None]
20. And(c != a, e != a, Or(c == e, bet(a, c, e), bet(a, e, c)))	[None]

True

In [205]:
# Lemma ang_add (Syntactic sugar for DA2): If d inside angle abc, then angle abd + angle dbc = angle abc.
pc = Proof( [interior(d,a,b,c)], Angle(a,b,c) == Angle(a,b,d) + Angle(d,b,c) )
pc.have( Distinct(a,b,c), 'ne_if_int' )
pc.construct('LC1', L, [a,b])
pc.construct('LC1', M, [b,c])
pc.have( And(ab==L, bc==M), 'G1' )
pc.have( L != M, 'ne_if_int' )
pc.have( Not(onLine(d,L)), 'S3' )
pc.have( Not(onLine(d,M)), 'S3' )
pc.have( And( sameside(d,a,M), sameside(d,c,L) ) )
pc.have(Angle(a,b,c) == Angle(a,b,d) + Angle(d,b,c) , 'DA2'  )
pc.qed('ang_add')

1. And(sameside(d, a, line(b, c)), sameside(d, c, line(a, b)))    / Angle(a, b, c) == Angle(a, b, d) + Angle(d, b, c)
--------------------
2. Distinct(a, b, c)	[ne_if_int]
3. Constructed L where And(onLine(a, L), onLine(b, L)) 	[LC1]
4. Constructed M where And(onLine(b, M), onLine(c, M)) 	[LC1]
5. And(line(a, b) == L, line(b, c) == M)	[G1]
6. L != M	[ne_if_int]
7. Not(onLine(d, L))	[S3]
8. Not(onLine(d, M))	[S3]
9. And(sameside(d, a, M), sameside(d, c, L))	[None]
10. Angle(a, b, c) == Angle(a, b, d) + Angle(d, b, c)	[DA2]
🔥 Success! You proved Angle(a, b, c) == Angle(a, b, d) + Angle(d, b, c) is true.
💾 Registered ang_add as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, d],
       Implies(And(sameside(d, a, line(b, c)),
                   sameside(d, c, line(a, b))),
               Angle(a, b, c) ==
               Angle(a, b, d) + Angle(d, b, c)))


True

In [222]:
# Lemma ang_post: Given distinct a b c, a d sameside bc, angle abc = angle dbc, then a d samray b
pc = Proof( [ Distinct(a,b,c), sameside(a,d,bc), Angle(a,b,c)==Angle(d,b,c) ], sameray(a,d,b) )
pc.construct('LC1', M, [b,a])
pc.have( ab==M, 'G1' )
pc.have( And( Not(onLine(a,bc)), Not(onLine(d,bc)) ), 'S3' )
pc.have( Not(onLine(c,M)), 'G1' )
pc.have( Or(onLine(d,M), sameside(d,c,M), diffside(d,c,M) ) )
pc.cases( Or(onLine(d,M), sameside(d,c,M), diffside(d,c,M) ) )
pc.have( coll(a,b,d), 'L4' )
pc.have( sameray(a,d,b),'sameray_if_sameside_coll' )
pc.close()
pc.have( interior(d,a,b,c) )
pc.have( Angle(a,b,c) == Angle(d,b,c) + Angle(a,b,d), 'ang_add' )
pc.have( Angle(d,b,c) == Angle(d,b,c) + Angle(a,b,d) )
pc.have( Angle(a,b,d) == 0 )
pc.have( onLine(d,M), 'DA1' )
pc.have( Not(onLine(d,M)), 'S3' )
pc.have(sameray(a,d,b) )  # From contradiction
pc.close()
pc.have( triangle(a,b,c), 'tri_if_non' )
pc.have( Implies( And( sameside(a,d,bc), diffside(c,d,M)), sameside(a,c,bd)), 'sameside_if_samediff' )
pc.have( interior(a,d,b,c) )
pc.have( Angle(d,b,c) == Angle(a,b,c) + Angle(a,b,d), 'ang_add' )
pc.have( Angle(a,b,d) == 0 )
pc.have( onLine(d,M), 'DA1' )
pc.have( Not(onLine(d,M)), 'S3' )
pc.have(sameray(a,d,b) )  # From contradiction
pc.close()
pc.qed()

1. Distinct(a, b, c)
2. sameside(a, d, line(b, c))
3. Angle(a, b, c) == Angle(d, b, c)    / And(a != b, d != b, Or(a == d, bet(b, a, d), bet(b, d, a)))
--------------------
4. Constructed M where And(onLine(b, M), onLine(a, M)) 	[LC1]
5. line(a, b) == M	[G1]
6. And(Not(onLine(a, line(b, c))), Not(onLine(d, line(b, c))))	[S3]
7. Not(onLine(c, M))	[G1]
8. Or(onLine(d, M),
   sameside(d, c, M),
   And(Not(onLine(d, M)),
       Not(onLine(c, M)),
       Not(sameside(d, c, M))))	[None]
9.1. Assuming case onLine(d, M). Try to reach the goal.
10. coll(a, b, d)	[L4]
11. And(a != b, d != b, Or(a == d, bet(b, a, d), bet(b, d, a)))	[sameray_if_sameside_coll]
📁 Case closed! onLine(d, M)'s goal reached.
9.2. Assuming case sameside(d, c, M). Try to reach the goal.
10. And(sameside(d, a, line(b, c)), sameside(d, c, line(a, b)))	[None]
11. Angle(a, b, c) == Angle(d, b, c) + Angle(a, b, d)	[ang_add]
12. Angle(d, b, c) == Angle(d, b, c) + Angle(a, b, d)	[None]
13. Angle(a, b, d) == 0	[None]
14. onLine(d

In [70]:
# Proposition 4: Given triangle abc and def, if ab=de, bc=ef, angle abc = angle def, then tri abc cong tri def.
# 'E4': ForAll([a,b,c,d,e,f], Implies(And(Segment(a,b)==Segment(d,e),Segment(b,c)==Segment(e,f), Angle(a,b,c)==Angle(d,e,f)  ),
#                                  cong(a,b,c,d,e,f) )),    #兩三角形兩邊角和夾角相應，則全等
pc = Proof( [triangle(a,b,c), triangle(d,e,f), Segment(a,b)==Segment(d,e),Segment(b,c)==Segment(e,f), Angle(a,b,c)==Angle(d,e,f) ],
                 cong(a,b,c,d,e,f) )
# 'SP': [ lambda a,b,c,d,g,h: And( Distinct(a,b,c), d!=g, Not(onLine(h,line(d,g))) ),
#         lambda x,y,z,a,b,c,d,g,h: And( x==d, sameray(y,g,d), sameside(z,h,line(d,g)), cong(x,y,z,a,b,c)   )  ]



1. Not(coll(a, b, c))
2. Not(coll(d, e, f))
3. Segment(a, b) == Segment(d, e)
4. Segment(b, c) == Segment(e, f)
5. Angle(a, b, c) == Angle(d, e, f)    / cong(a, b, c, d, e, f)
--------------------


In [19]:
# Proposition 5: Given distinct a b c, ab = ac, a-b-d, a-c-e, prove ∠abc = ∠acb, ∠cbd = ∠bce  (base angs, isos. tri)
pc = Proof([ Distinct(a,b,c), Segment(a,b) == Segment(a,c)], Angle(a,b,c) == Angle(a,c,b)  )
pc.have( Angle(a,b,c) == Angle(c,b,a), 'M4')
pc.have(  cong(b,a,c,c,a,b), 'E4' )
pc.have( Angle(a,b,c) == Angle(a,c,b), 'CG' )
pc.qed('E5')

1. Distinct(a, b, c)
2. Segment(a, b) == Segment(a, c)    / Angle(a, b, c) == Angle(a, c, b)
--------------------
3. Angle(a, b, c) == Angle(c, b, a)	[M4]
4. cong(b, a, c, c, a, b)	[E4]
5. Angle(a, b, c) == Angle(a, c, b)	[CG]
🔥 Success! You proved Angle(a, b, c) == Angle(a, c, b) is true.
💾 Registered E5 as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(And(Distinct(a, b, c),
                   Segment(a, b) == Segment(a, c)),
               Angle(a, b, c) == Angle(a, c, b)))


True

In [379]:
# Proposition 5a: Given distinct a b c, ab = ac, a-b-d, a-c-e, prove ∠abc = ∠acb, ∠cbd = ∠bce.  (Elements version)
pc = Proof([ Distinct(a,b,c), Segment(a,b) == Segment(a,c), bet(a,b,d), bet(a,c,e) ], 
           And(Angle(a,b,c) == Angle(a,c,b), Angle(c,b,d) == Angle(b,c,e))  )
pc.have( Angle(a,b,c) == Angle(c,b,a), 'M4')
pc.have(  cong(b,a,c,c,a,b), 'E4' )
pc.have( Angle(a,b,c) == Angle(a,c,b) )
pc.construct('CP4', f, [a,b,ab]);
pc.construct('LC2', α, [a,f])
pc.have(inside(a,α), 'G3')
pc.have( Segment(a,b) + Segment(b,f) == Segment(a,f), 'DS1' )
pc.have( Segment(b,f) > 0, 'M1' )
pc.have( Segment(a,b) < Segment(a,f) )
pc.have( Segment(a,c) < Segment(a,f) )
pc.have( inside(c,α), 'DS4' )
pc.construct('CI5', g, [c,a,ac,α])
pc.have( Segment(a,f) == Segment(a,g), 'DS3' )
# pc.have( Implies(And(Segment(a, b) == Segment(d, e),
#                    Segment(b, c) == Segment(e, f),
#                    Angle(a, b, c) == Angle(d, e, f)),
#                cong(a,b,c,d,e,f)), 'E4' )
pc.have( And(coll(a,b,f), coll(a,c,g), b != a, f != a, c != a, g != a, Not(bet(b, a, f)), )  )
pc.have( Angle(f,a,c) == Angle(g,a,b), 'DA4' )
pc.have( cong(f,a,c,g,a,b), 'E4' )
pc.have( Angle(a,f,c) == Angle(a,g,b), 'CG' )
pc.have( Segment(f,c) == Segment(b,g), 'CG' )
# pc.have( Implies(And(coll(a, b, e),
#                    coll(a, c, f),
#                    b != a,
#                    e != a,
#                    c != a,
#                    f != a,
#                    Not(bet(b, a, e)),
#                    Not(bet(c, a, f))),
#                Angle(b, a, c) == Angle(e, a, f)), 'DA4' )
pc.have( sameray(f,b,a) )
pc.have( sameray(c,c,f) )
#pc.have( And(coll(f,b,a), coll(f,c,c), a != f, b != f, c != f, c != f, Not(bet(a, f, b)), Not(bet(c,f,c)) )  )
pc.have( Angle(a,f,c) == Angle(b,f,c), 'eqang_if_sameray' )
pc.have( And(coll(g,c,a), coll(g,b,b), a != g, c != g, b != g, b != f, Not(bet(a, g, c)), Not(bet(b,g,b)) )  )
pc.have( Angle(a,g,b) == Angle(c,g,b), 'DA4' )
pc.have( Angle(b,f,c) == Angle(c,g,b), 'DA4' )
pc.have( Segment(a,c) + Segment(c,g) == Segment(a,g), 'DS1' )
pc.have( Segment(b,f) == Segment(c,g), 'CG' )
pc.have( cong(b,f,c,c,g,b), 'E4' )
pc.have( Angle(f,b,c) == Angle(g,c,b), 'CG' )
pc.have( And(bet(a,b,d),bet(a,b,f)) )
pc.have( Not(bet(d,b,f)), 'B7' )
pc.have( And(bet(a,b,d), onLine(a,ab), onLine(b,ab) ) )
pc.have( onLine(d,ab), 'B2' )
pc.have( And(coll(b,d,f), coll(b,c,c), f != b, d != b, c != b, c != b, Not(bet(d,b,f)), Not(bet(c,b,c)) ), 'L4'  )
pc.have( Angle(d,b,c) == Angle(f,b,c), 'DA4' )

pc.have( Not(bet(e,c,g)), 'B7' )
pc.have( And(bet(a,c,e), onLine(a,ac), onLine(c,ac) ) )
pc.have( onLine(e,ac), 'B2' )
pc.have( And(coll(c,e,g), coll(c,b,b)), 'L4' )
pc.have( And( g != c, e != c, b != c, b != c, Not(bet(e,c,g)), Not(bet(b,c,b)) )  )
pc.have( Angle(e,c,b) == Angle(g,c,b), 'DA4' )
pc.have( Angle(c,b,d) == Angle(b,c,e) )
pc.have( And(Angle(a,b,c) == Angle(a,c,b), Angle(c,b,d) == Angle(b,c,e)), 'CG' )
pc.qed('E5a')

1. Distinct(a, b, c)
2. Segment(a, b) == Segment(a, c)
3. bet(a, b, d)
4. bet(a, c, e)    / And(Angle(a, b, c) == Angle(a, c, b),
    Angle(c, b, d) == Angle(b, c, e))
--------------------
5. Angle(a, b, c) == Angle(c, b, a)	[M4]
6. cong(b, a, c, c, a, b)	[E4]
7. Angle(a, b, c) == Angle(a, c, b)	[None]
8. Constructed f where And(onLine(f, line(a, b)), bet(a, b, f)) 	[CP4]
9. Constructed α where And(center(a, α), onCircle(f, α)) 	[LC2]
10. inside(a, α)	[G3]
11. Segment(a, b) + Segment(b, f) == Segment(a, f)	[DS1]
12. Segment(b, f) > 0	[M1]
13. Segment(a, b) < Segment(a, f)	[None]
14. Segment(a, c) < Segment(a, f)	[None]
15. inside(c, α)	[DS4]
16. Constructed g where And(onCircle(g, α), onLine(g, line(a, c)), bet(g, c, a)) 	[CI5]
17. Segment(a, f) == Segment(a, g)	[DS3]
18. And(coll(a, b, f),
    coll(a, c, g),
    b != a,
    f != a,
    c != a,
    g != a,
    Not(bet(b, a, f)))	[None]
19. Angle(f, a, c) == Angle(g, a, b)	[DA4]
20. cong(f, a, c, g, a, b)	[E4]
21. Angle(a, f, c) == Angl

True

In [381]:
# Aux 6a (proof by contradiction)
pc = Proof( [triangle(a,b,c), Angle(a,b,c) == Angle(a,c,b), Segment(a,b) > Segment(a,c)  ], Segment(a,b) == Segment(a,c) )
pc.construct('E3', d, [b,a,a,c])
pc.have( Angle(d,b,c) == Angle(a,b,c), 'eqang_if_sameray' )
pc.have( Angle(d,b,c) == Angle(a,c,b) )
pc.have( And( Segment(d,b)==Segment(a,c), Segment(b,c)==Segment(c,b) ) )
pc.have( cong(d,b,c,a,c,b), 'E4' )
pc.have( Segment(d,c) == Segment(a,b), 'CG' )
pc.have( Segment(d,c) > Segment(a,c) )
pc.have( Segment(a,d) + Segment(d,b) == Segment(a,b), 'DS1' )
pc.have( Area(d,b,c) == Area(a,c,b), 'M9' )
pc.have( onLine(d,ab), 'B3' )
pc.have( bet(b,d,a) )
pc.have( Not(onLine(c,ab)), 'non_if_tri' )
pc.have( Area(a,c,b) == Area(a,d,c) + Area(c,d,b), 'DR2')
pc.have(  Area(a,d,c) > 0, 'DR1' )
pc.have(  Area(a,c,b) > Area(c,d,b))
pc.have( Area(c,d,b) == Area(d,b,c), 'M8' )
pc.have(  Area(a,c,b) > Area(d,b,c) )
pc.have( And(  Area(a,c,b) == Area(d,b,c), Area(a,c,b) > Area(d,b,c), Segment(a,b) == Segment(a,c)  ) )
pc.have( Segment(a,b) == Segment(a,c) )
pc.qed('A6a')

1. Not(coll(a, b, c))
2. Angle(a, b, c) == Angle(a, c, b)
3. Segment(a, b) > Segment(a, c)    / Segment(a, b) == Segment(a, c)
--------------------
4. Constructed d where And(bet(b, d, a), Segment(b, d) == Segment(a, c)) 	[E3]
5. Angle(d, b, c) == Angle(a, b, c)	[eqang_if_sameray]
6. Angle(d, b, c) == Angle(a, c, b)	[None]
7. And(Segment(d, b) == Segment(a, c),
    Segment(b, c) == Segment(c, b))	[None]
8. cong(d, b, c, a, c, b)	[E4]
9. Segment(d, c) == Segment(a, b)	[CG]
10. Segment(d, c) > Segment(a, c)	[None]
11. Segment(a, d) + Segment(d, b) == Segment(a, b)	[DS1]
12. Area(d, b, c) == Area(a, c, b)	[M9]
13. onLine(d, line(a, b))	[B3]
14. bet(b, d, a)	[None]
15. Not(onLine(c, line(a, b)))	[non_if_tri]
16. Area(a, c, b) == Area(a, d, c) + Area(c, d, b)	[D2]
17. Area(a, d, c) > 0	[D1]
18. Area(a, c, b) > Area(c, d, b)	[None]
19. Area(c, d, b) == Area(d, b, c)	[M8]
20. Area(a, c, b) > Area(d, b, c)	[None]
21. And(Area(a, c, b) == Area(d, b, c),
    Area(a, c, b) > Area(d, b, c),
    Se

True

In [382]:
# Aux 6b (symmetrically from 6a)
pc = Proof( [triangle(a,b,c), Angle(a,b,c) == Angle(a,c,b), Segment(a,b) < Segment(a,c)  ], Segment(a,b) == Segment(a,c) )
pc.have( triangle(a,c,b) )
pc.have( Angle(a,c,b) == Angle(a,b,c) )
pc.have( Segment(a,c) > Segment(a,b) )
pc.have( Segment(a,c) == Segment(a,b), 'A6a' )
pc.have( Segment(a,b) == Segment(a,c) )
pc.qed('A6b')

1. Not(coll(a, b, c))
2. Angle(a, b, c) == Angle(a, c, b)
3. Segment(a, b) < Segment(a, c)    / Segment(a, b) == Segment(a, c)
--------------------
4. Not(coll(a, c, b))	[None]
5. Angle(a, c, b) == Angle(a, b, c)	[None]
6. Segment(a, c) > Segment(a, b)	[None]
7. Segment(a, c) == Segment(a, b)	[A6a]
8. Segment(a, b) == Segment(a, c)	[None]
🔥 Success! You proved Segment(a, b) == Segment(a, c) is true.
💾 Registered A6b as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(And(Not(coll(a, b, c)),
                   Angle(a, b, c) == Angle(a, c, b),
                   Segment(a, b) < Segment(a, c)),
               Segment(a, b) == Segment(a, c)))


True

In [383]:
# Proposition 6: Given triangle abc with ∠abc = ∠acb, prove ab = ac  (sides opp. angles equal)
pc = Proof( [triangle(a,b,c), Angle(a,b,c) == Angle(a,c,b) ], Segment(a,b) == Segment(a,c) )
pc.have( Or( Segment(a,b) > Segment(a,c), Segment(a,b) < Segment(a,c), Segment(a,b) == Segment(a,c) )  )
pc.cases( Or( Segment(a,b) > Segment(a,c), Segment(a,b) < Segment(a,c), Segment(a,b) == Segment(a,c) ) )
pc.have( Segment(a,b) == Segment(a,c), 'A6a' )
pc.close()
pc.have( Segment(a,b) == Segment(a,c), 'A6b' )
pc.close()
pc.close()
pc.qed('E6')

1. Not(coll(a, b, c))
2. Angle(a, b, c) == Angle(a, c, b)    / Segment(a, b) == Segment(a, c)
--------------------
3. Or(Segment(a, b) > Segment(a, c),
   Segment(a, b) < Segment(a, c),
   Segment(a, b) == Segment(a, c))	[None]
4.1. Assuming case Segment(a, b) > Segment(a, c). Try to reach the goal.
5. Segment(a, b) == Segment(a, c)	[A6a]
📁 Case closed! Segment(a, b) > Segment(a, c)'s goal reached.
4.2. Assuming case Segment(a, b) < Segment(a, c). Try to reach the goal.
5. Segment(a, b) == Segment(a, c)	[A6b]
📁 Case closed! Segment(a, b) < Segment(a, c)'s goal reached.
4.3. Assuming case Segment(a, b) == Segment(a, c). Try to reach the goal.
📁 Case closed! Segment(a, b) == Segment(a, c)'s goal reached.
All cases closed! Segment(a, b) == Segment(a, c) have been added to context.
🔥 Success! You proved Segment(a, b) == Segment(a, c) is true.
💾 Registered E6 as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(And(Not(coll(a, b, c)),
                   Angle(a, b, c)

True

In [384]:
# Lemma ne_if_ins: A point inside a triangle is distinct from the vertices.
pc = Proof( [insideT(p,a,b,c)], Distinct(p,a,b,c) )
pc.have(Not(onLine(a, bc)), rule='S3')
pc.have(Not(onLine(b, ac)), rule='S3')
pc.have(Not(onLine(c, ab)), rule='S3')
pc.have(Distinct(a, b, c) )
pc.have( Not( onLine(p,ac) ), 'S3' )
pc.have( Not( onLine(p,ab) ), 'S3' )
pc.have( Not( onLine(p,bc) ), 'S3' )
pc.have( And(p !=a, p!=b, p!=c) )
pc.have( Distinct(p,a,b,c))
pc.qed('ne_if_ins')

1. And(sameside(p, a, line(b, c)),
    sameside(p, b, line(a, c)),
    sameside(p, c, line(a, b)))    / Distinct(p, a, b, c)
--------------------
2. Not(onLine(a, line(b, c)))	[S3]
3. Not(onLine(b, line(a, c)))	[S3]
4. Not(onLine(c, line(a, b)))	[S3]
5. Distinct(a, b, c)	[None]
6. Not(onLine(p, line(a, c)))	[S3]
7. Not(onLine(p, line(a, b)))	[S3]
8. Not(onLine(p, line(b, c)))	[S3]
9. And(p != a, p != b, p != c)	[None]
10. Distinct(p, a, b, c)	[None]
🔥 Success! You proved Distinct(p, a, b, c) is true.
💾 Registered ne_if_ins as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, p],
       Implies(And(sameside(p, a, line(b, c)),
                   sameside(p, b, line(a, c)),
                   sameside(p, c, line(a, b))),
               Distinct(p, a, b, c)))


True

In [385]:
# lemma tri_if_ins: If there exists point p inside triangle abc, then abc forms a triangle.
pc = Proof( [insideT(p,a,b,c)], triangle(a,b,c) )
pc.have( Distinct(p,a,b,c), 'ne_if_ins' )
pc.have( Not(onLine(c, ab)), 'S3' )
pc.have( ab != ac )
pc.have( Not(bet(a, b, c)), 'B2' )
pc.have( Not(bet(b, a, c)), 'B2' )
pc.have( Not(bet(a, c, b)), 'B3' )
# 5. All betweenness cases are eliminated, so non-collinearity holds!
pc.have( Not(coll(a, b, c)) )
pc.have( triangle(a,b,c) )
pc.qed('tri_if_ins')

1. And(sameside(p, a, line(b, c)),
    sameside(p, b, line(a, c)),
    sameside(p, c, line(a, b)))    / Not(coll(a, b, c))
--------------------
2. Distinct(p, a, b, c)	[ne_if_ins]
3. Not(onLine(c, line(a, b)))	[S3]
4. line(a, b) != line(a, c)	[None]
5. Not(bet(a, b, c))	[B2]
6. Not(bet(b, a, c))	[B2]
7. Not(bet(a, c, b))	[B3]
8. Not(coll(a, b, c))	[None]
--> #9 have(Not(coll(a, b, c)), None): You have proved Not(coll(a, b, c)) before. Try again.
🔥 Success! You proved Not(coll(a, b, c)) is true.
💾 Registered tri_if_ins as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, p],
       Implies(And(sameside(p, a, line(b, c)),
                   sameside(p, b, line(a, c)),
                   sameside(p, c, line(a, b))),
               Not(coll(a, b, c))))


True

In [386]:
# Lemma Tdiff: If bet(a,b,c), b on L, a ~on L, then ac diffside L.
pc = Proof( [bet(a,b,c), onLine(b,L), Not(onLine(a,L))], diffside(a,c,L))
pc.have(bet(c, b, a), 'B1')
pc.have( Not(onLine(c,L)), 'B2' )
pc.have(Not(sameside(a, c, L)), 'P3')
pc.have( diffside(a,c,L) )
pc.qed('Tdiff')

1. bet(a, b, c)
2. onLine(b, L)
3. Not(onLine(a, L))    / And(Not(onLine(a, L)),
    Not(onLine(c, L)),
    Not(sameside(a, c, L)))
--------------------
4. bet(c, b, a)	[B1]
5. Not(onLine(c, L))	[B2]
6. Not(sameside(a, c, L))	[P3]
7. And(Not(onLine(a, L)),
    Not(onLine(c, L)),
    Not(sameside(a, c, L)))	[None]
🔥 Success! You proved And(Not(onLine(a, L)),
    Not(onLine(c, L)),
    Not(sameside(a, c, L))) is true.
💾 Registered Tdiff as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, L],
       Implies(And(bet(a, b, c),
                   onLine(b, L),
                   Not(onLine(a, L))),
               And(Not(onLine(a, L)),
                   Not(onLine(c, L)),
                   Not(sameside(a, c, L)))))


True

In [388]:
# Lemma diffdiff_aux: If a b diffside cd, c d diffside ab, then a c sameside bd and a d sameside bc
pc = Proof( [diffside(a,b,cd), diffside(c,d,ab)], And(sameside(a,c,bd), sameside(a,d,bc)) )
pc.have( interLL(ab,cd), 'I1' )
pc.construct('CI1', e, [ab,cd])
pc.have(Not(onLine(a, cd)), 'S3')
pc.have(e != a )
pc.have(Not(onLine(b, cd)), 'S3')
pc.have(e != b )
pc.have(Not(onLine(c, ab)), 'S3')
pc.have(e != c )
pc.have(Not(onLine(d, ab)), 'S3')
pc.have(e != d )
pc.have( a!=b, 'ne_if_diff' )
pc.have( c!=d, 'ne_if_diff' )
pc.have(Distinct(a, e, b, c, d) )
pc.have( bet(a, e, b), 'P4' )
pc.have( bet(c, e, d), 'P4' )
pc.have( bet(b, e, a) )
pc.have( onLine(b,bd) )
pc.have(Implies(onLine(e, bd), ab == bd), 'G1')
pc.have(Not(onLine(e, bd)))
pc.have( sameside(e,a,bd), 'P2' )
pc.have( sameside(e,c,bd), 'P2' )
pc.have( sameside(a,c,bd), 'S4' )
pc.have( onLine(b,bc) )
pc.have(Implies(onLine(e, bc), ab == bc), 'G1')
pc.have(Not(onLine(e, bc)))
pc.have( sameside(e,d,bc), 'P2' )
pc.have( sameside(e,a,bc), 'P2' )
pc.have( sameside(a,d,bc), 'S4' )
pc.have( And(sameside(a,c,bd), sameside(a,d,bc)) )
pc.qed('diffdiff_aux')

1. And(Not(onLine(a, line(c, d))),
    Not(onLine(b, line(c, d))),
    Not(sameside(a, b, line(c, d))))
2. And(Not(onLine(c, line(a, b))),
    Not(onLine(d, line(a, b))),
    Not(sameside(c, d, line(a, b))))    / And(sameside(a, c, line(b, d)), sameside(a, d, line(b, c)))
--------------------
3. interLL(line(a, b), line(c, d))	[I1]
4. Constructed e where And(onLine(e, line(a, b)), onLine(e, line(c, d))) 	[CI1]
5. Not(onLine(a, line(c, d)))	[S3]
6. e != a	[None]
7. Not(onLine(b, line(c, d)))	[S3]
8. e != b	[None]
9. Not(onLine(c, line(a, b)))	[S3]
10. e != c	[None]
11. Not(onLine(d, line(a, b)))	[S3]
12. e != d	[None]
13. a != b	[ne_if_diff]
14. c != d	[ne_if_diff]
15. Distinct(a, e, b, c, d)	[None]
16. bet(a, e, b)	[P4]
17. bet(c, e, d)	[P4]
18. bet(b, e, a)	[None]
19. onLine(b, line(b, d))	[None]
20. Implies(onLine(e, line(b, d)), line(a, b) == line(b, d))	[G1]
21. Not(onLine(e, line(b, d)))	[None]
22. sameside(e, a, line(b, d))	[P2]
23. sameside(e, c, line(b, d))	[P2]
24. sameside(a,

True

In [389]:
# Lemma diffdiff: If a b diffside cd, c d diffside ab, then a c sameside bd and a d sameside bc
pc = Proof( [diffside(a,b,cd), diffside(c,d,ab)], And(sameside(a,c,bd), sameside(a,d,bc), sameside(b,c,ad), sameside(b,d,ac) ) )
pc.have( And(sameside(a,c,bd), sameside(a,d,bc)), 'diffdiff_aux' )
pc.have( And( sameside(b,c,ad), sameside(b,d,ac) ), 'diffdiff_aux' )
pc.have( And(sameside(a,c,bd), sameside(a,d,bc), sameside(b,c,ad), sameside(b,d,ac) ) )
pc.qed('diffdiff')

1. And(Not(onLine(a, line(c, d))),
    Not(onLine(b, line(c, d))),
    Not(sameside(a, b, line(c, d))))
2. And(Not(onLine(c, line(a, b))),
    Not(onLine(d, line(a, b))),
    Not(sameside(c, d, line(a, b))))    / And(sameside(a, c, line(b, d)),
    sameside(a, d, line(b, c)),
    sameside(b, c, line(a, d)),
    sameside(b, d, line(a, c)))
--------------------
3. And(sameside(a, c, line(b, d)), sameside(a, d, line(b, c)))	[diffdiff_aux]
4. And(sameside(b, c, line(a, d)), sameside(b, d, line(a, c)))	[diffdiff_aux]
5. And(sameside(a, c, line(b, d)),
    sameside(a, d, line(b, c)),
    sameside(b, c, line(a, d)),
    sameside(b, d, line(a, c)))	[None]
🔥 Success! You proved And(sameside(a, c, line(b, d)),
    sameside(a, d, line(b, c)),
    sameside(b, c, line(a, d)),
    sameside(b, d, line(a, c))) is true.
💾 Registered diffdiff as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, d],
       Implies(And(And(Not(onLine(a, line(c, d))),
                       Not(onLine(b, line(c, d))

True

In [390]:
# Lemma cross: If a b diffside cd, c d diffside ab, then a c sameside bd and a d sameside bc
pc = Proof( [ bet(a,e,b), bet(c,e,d), Not(onLine(c,ab)) ], And(sameside(a,c,bd), sameside(a,d,bc), sameside(b,c,ad), sameside(b,d,ac)) )
pc.construct('LC1', L, [a,b])
pc.construct('LC1', M, [c,d])
pc.have( And(onLine(e,L), onLine(e,M)), 'B3' )
pc.have( Not(onLine(a,M), onLine(b,M)), 'G1' )
pc.have( Not(sameside(a,b,M)), 'P3' )
pc.have( diffside(a,b,M), 'G1' )
pc.have( Not(sameside(c,d,L)), 'P3' )
pc.have( diffside(c,d,L), 'G1' )
pc.have( diffside(a,b,cd), 'G1' )
pc.have( diffside(c,d,ab), 'G1' )
pc.have( And(sameside(a,c,bd), sameside(a,d,bc), sameside(b,c,ad), sameside(b,d,ac)), 'diffdiff' )
pc.qed('cross')

1. bet(a, e, b)
2. bet(c, e, d)
3. Not(onLine(c, line(a, b)))    / And(sameside(a, c, line(b, d)),
    sameside(a, d, line(b, c)),
    sameside(b, c, line(a, d)),
    sameside(b, d, line(a, c)))
--------------------
4. Constructed L where And(onLine(a, L), onLine(b, L)) 	[LC1]
5. Constructed M where And(onLine(c, M), onLine(d, M)) 	[LC1]
6. And(onLine(e, L), onLine(e, M))	[B3]
7. Not(onLine(a, M))	[G1]
8. Not(sameside(a, b, M))	[P3]
9. And(Not(onLine(a, M)),
    Not(onLine(b, M)),
    Not(sameside(a, b, M)))	[G1]
10. Not(sameside(c, d, L))	[P3]
11. And(Not(onLine(c, L)),
    Not(onLine(d, L)),
    Not(sameside(c, d, L)))	[G1]
12. And(Not(onLine(a, line(c, d))),
    Not(onLine(b, line(c, d))),
    Not(sameside(a, b, line(c, d))))	[G1]
13. And(Not(onLine(c, line(a, b))),
    Not(onLine(d, line(a, b))),
    Not(sameside(c, d, line(a, b))))	[G1]
14. And(sameside(a, c, line(b, d)),
    sameside(a, d, line(b, c)),
    sameside(b, c, line(a, d)),
    sameside(b, d, line(a, c)))	[diffdiff]
🔥 S

True

In [392]:
# Lemma lt_if_int: If p inside angle abc, then angle abp < angle abc && cbp < angle abc, and angle addition rule holds.
pc = Proof( [interior(p,a,b,c)], And( Angle(a, b, c) == Angle(a, b, p) + Angle(p, b, c), 
                                      Angle(a,b,p) < Angle(a,b,c), Angle(c,b,p) < Angle(a,b,c) ) )
pc.have( Not(onLine(a,bc)) , 'S3' )
pc.have( Not(onLine(c,ab)) , 'S3' )
pc.have( And(b!=a, b!=c ) )
pc.have( ab != bc, 'ne_if_int' )
pc.have( Not(onLine(p, ab)), 'S3' )
pc.have(Not(onLine(p, bc)), 'S3')
pc.have( Angle(a, b, c) == Angle(a, b, p) + Angle(p, b, c), 'DA2'  )
pc.have( Angle(p, b, c) != 0, 'DA1' )
pc.have( Angle(c,b,p) < Angle(a,b,c) )
pc.have( Angle(a,b,p) != 0, 'DA1' )
pc.have( Angle(a,b,p) < Angle(a,b,c) )
pc.have( And( Angle(a, b, c) == Angle(a, b, p) + Angle(p, b, c), 
                                      Angle(a,b,p) < Angle(a,b,c), Angle(c,b,p) < Angle(a,b,c) )  )
pc.qed('lt_if_int')

1. And(sameside(p, a, line(b, c)), sameside(p, c, line(a, b)))    / And(Angle(a, b, c) == Angle(a, b, p) + Angle(p, b, c),
    Angle(a, b, p) < Angle(a, b, c),
    Angle(c, b, p) < Angle(a, b, c))
--------------------
2. Not(onLine(a, line(b, c)))	[S3]
3. Not(onLine(c, line(a, b)))	[S3]
4. And(b != a, b != c)	[None]
5. line(a, b) != line(b, c)	[ne_if_int]
6. Not(onLine(p, line(a, b)))	[S3]
7. Not(onLine(p, line(b, c)))	[S3]
8. Angle(a, b, c) == Angle(a, b, p) + Angle(p, b, c)	[DA2]
9. Angle(p, b, c) != 0	[DA1]
--> #10 have(Angle(c, b, p) < Angle(a, b, c), None): Sorry, can't infer Angle(c, b, p) < Angle(a, b, c) from None.
10. Angle(a, b, p) != 0	[DA1]
11. Angle(a, b, p) < Angle(a, b, c)	[None]
12. And(Angle(a, b, c) == Angle(a, b, p) + Angle(p, b, c),
    Angle(a, b, p) < Angle(a, b, c),
    Angle(c, b, p) < Angle(a, b, c))	[None]
🔥 Success! You proved And(Angle(a, b, c) == Angle(a, b, p) + Angle(p, b, c),
    Angle(a, b, p) < Angle(a, b, c),
    Angle(c, b, p) < Angle(a, b, c)) is tr

True

In [393]:
# Lemma diff_if_int: If p is inside angle abc, then a c diffside bp.
pc = Proof( [interior(p,a,b,c)], diffside(a,c,bp) )
pc.have( And( onLine(b,bc), onLine(b,ab), onLine(b,bp), onLine(a,ab), onLine(c,bc), onLine(p,bp), sameside(c,p,ab) ) )
pc.have( Not(sameside(a,c,bp)), 'T1' )
# 1. Extract the non-incidence facts showing p is not on the angle arms
pc.have(Not(onLine(p, ab)), rule='S3')
pc.have(Not(onLine(p, bc)), rule='S3')
# 2. Handle the non-incidence of 'a' on line bp via G1 collapse
pc.have( Distinct(ab, bc, ac), 'ne_if_int' )
pc.have(Not(onLine(a, bp)), 'G1' )
pc.have(Not(onLine(c, bp)), 'G1' )
pc.have( diffside(a,c,bp) )
pc.qed('diff_if_int')

1. And(sameside(p, a, line(b, c)), sameside(p, c, line(a, b)))    / And(Not(onLine(a, line(b, p))),
    Not(onLine(c, line(b, p))),
    Not(sameside(a, c, line(b, p))))
--------------------
2. And(onLine(b, line(b, c)),
    onLine(b, line(a, b)),
    onLine(b, line(b, p)),
    onLine(a, line(a, b)),
    onLine(c, line(b, c)),
    onLine(p, line(b, p)),
    sameside(c, p, line(a, b)))	[None]
3. Not(sameside(a, c, line(b, p)))	[T1]
4. Not(onLine(p, line(a, b)))	[S3]
5. Not(onLine(p, line(b, c)))	[S3]
6. Distinct(line(a, b), line(b, c), line(a, c))	[ne_if_int]
7. Not(onLine(a, line(b, p)))	[G1]
8. Not(onLine(c, line(b, p)))	[G1]
9. And(Not(onLine(a, line(b, p))),
    Not(onLine(c, line(b, p))),
    Not(sameside(a, c, line(b, p))))	[None]
🔥 Success! You proved And(Not(onLine(a, line(b, p))),
    Not(onLine(c, line(b, p))),
    Not(sameside(a, c, line(b, p)))) is true.
💾 Registered diff_if_int as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, p],
       Implies(And(sameside(p, a, 

True

In [395]:
# Lemma R2a: If b c sameside L, a on L, a b c collinear, then b c sameside L
pc = Proof( [ sameside(b,c,L), onLine(a,L), coll(a,b,c) ], sameray(b,c,a) )
pc.have( And(b!=a, c!=a), 'S3' )
pc.have( Or( bet(a,b,c), bet(b,c,a), bet(c,a,b), b==c ), 'L0' )
pc.have(Not(bet(b,a,c)), 'P3')
pc.have( sameray(b,c,a) )
pc.qed( 'R2a' )

1. sameside(b, c, L)
2. onLine(a, L)
3. coll(a, b, c)    / And(b != a, c != a, Or(b == c, bet(a, b, c), bet(a, c, b)))
--------------------
4. And(b != a, c != a)	[S3]
5. Or(bet(a, b, c), bet(b, c, a), bet(c, a, b), b == c)	[L0]
6. Not(bet(b, a, c))	[P3]
7. And(b != a, c != a, Or(b == c, bet(a, b, c), bet(a, c, b)))	[None]
🔥 Success! You proved And(b != a, c != a, Or(b == c, bet(a, b, c), bet(a, c, b))) is true.
💾 Registered R2a as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, L],
       Implies(And(sameside(b, c, L),
                   onLine(a, L),
                   coll(a, b, c)),
               And(b != a,
                   c != a,
                   Or(b == c, bet(a, b, c), bet(a, c, b)))))


True

In [396]:
# Lemma int_sameray_if_int: If p is inside angle abc, and bd,be sameray ba,bc respectively, then p inside angle dbe.
pc = Proof( [interior(p,a,b,c), sameray(a,d,b), sameray(c,e,b)], interior(p,d,b,e) )
pc.construct('CL', L, [a,d,b])
pc.construct('CL', M, [c,e,b])
pc.have( Angle(a,b,c) == Angle(d,b,e), 'eqang_if_sameray' )
pc.have( ab == L, 'G1' )
pc.have( bd == L, 'G1' )
pc.have( bc == M, 'G1' )
pc.have( be == M, 'G1' )
pc.have( And( sameside(p,c,L), sameside(p,a,M)) )
pc.have( L!=M, 'ne_if_int' )
pc.have( Not(onLine(a,M)), 'G1' )
pc.have( Not(onLine(d,M)), 'G1' )
pc.have( Not(onLine(c,L)), 'G1' )
pc.have( Not(onLine(e,L)), 'G1' )
# [sameray(b,c,a), onLine(a,L), Not(onLine(b,L)) ] => sameside(b,c,L)
pc.have( sameside(a,d,M), 'R2' )
pc.have( sameside(c,e,L), 'R2' )
pc.have( sameside(p,d,M), 'S4' )
pc.have( sameside(p,e,L), 'S4' )
pc.have( interior(p,d,b,e) )
pc.qed('int_sameray_if_int')

1. And(sameside(p, a, line(b, c)), sameside(p, c, line(a, b)))
2. And(a != b, d != b, Or(a == d, bet(b, a, d), bet(b, d, a)))
3. And(c != b, e != b, Or(c == e, bet(b, c, e), bet(b, e, c)))    / And(sameside(p, d, line(b, e)), sameside(p, e, line(d, b)))
--------------------
4. Constructed L where And(onLine(a, L), onLine(d, L), onLine(b, L)) 	[CL]
5. Constructed M where And(onLine(c, M), onLine(e, M), onLine(b, M)) 	[CL]
6. Angle(a, b, c) == Angle(d, b, e)	[eqang_if_sameray]
7. line(a, b) == L	[G1]
8. line(b, d) == L	[G1]
9. line(b, c) == M	[G1]
10. line(b, e) == M	[G1]
11. And(sameside(p, c, L), sameside(p, a, M))	[None]
12. L != M	[ne_if_int]
13. Not(onLine(a, M))	[G1]
14. Not(onLine(d, M))	[G1]
15. Not(onLine(c, L))	[G1]
16. Not(onLine(e, L))	[G1]
17. sameside(a, d, M)	[R2]
18. sameside(c, e, L)	[R2]
19. sameside(p, d, M)	[S4]
20. sameside(p, e, L)	[S4]
21. And(sameside(p, d, line(b, e)), sameside(p, e, line(d, b)))	[None]
🔥 Success! You proved And(sameside(p, d, line(b, e)), samesi

True

In [398]:
# Lemma on_if_coll: Given points b != c, a not on bc is equivalent to a b c not collinear.
pc = Proof( [ b != c, coll(a,b,c)], onLine(a,bc) )
pc.construct('CL', L, [a,b,c])
pc.have( L == bc, 'G1' )
pc.have( onLine(a,bc)  )
pc.qed('on_if_coll')

1. b != c
2. coll(a, b, c)    / onLine(a, line(b, c))
--------------------
3. Constructed L where And(onLine(a, L), onLine(b, L), onLine(c, L)) 	[CL]
4. L == line(b, c)	[G1]
5. onLine(a, line(b, c))	[None]
🔥 Success! You proved onLine(a, line(b, c)) is true.
💾 Registered on_if_coll as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(And(b != c, coll(a, b, c)),
               onLine(a, line(b, c))))


True

In [399]:
# Lemma on_iff_coll: Given points b != c, a not on bc is equivalent to a b c not collinear.
pc = Proof( [ points(a), b != c], onLine(a,bc) == coll(a,b,c) )
pc.have( Implies(onLine(a,bc), coll(a,b,c)), 'L4' )
pc.have( Implies( coll(a,b,c), Or( bet(a,b,c), bet(b,c,a), bet(c,a,b), a==b, a==c ) ) )
pc.have( Implies(coll(a,b,c), onLine(a,bc)), 'on_if_coll' )
pc.have( onLine(a,bc) == coll(a,b,c) )
pc.qed('on_iff_coll')

1. Or(Distinct(a), Not(Distinct(a)))
2. b != c    / onLine(a, line(b, c)) == coll(a, b, c)
--------------------
3. Implies(onLine(a, line(b, c)), coll(a, b, c))	[L4]
4. Implies(coll(a, b, c),
        Or(bet(a, b, c),
           bet(b, c, a),
           bet(c, a, b),
           a == b,
           a == c))	[None]
5. Implies(coll(a, b, c), onLine(a, line(b, c)))	[on_if_coll]
6. onLine(a, line(b, c)) == coll(a, b, c)	[None]
🔥 Success! You proved onLine(a, line(b, c)) == coll(a, b, c) is true.
💾 Registered on_iff_coll as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c],
       Implies(And(Or(Distinct(a), Not(Distinct(a))),
                   b != c),
               onLine(a, line(b, c)) == coll(a, b, c)))


True

In [400]:
# Lemma nbet_if_side: If a c sameside L, b on L, then b is not between a and c.
pc = Proof( [sameside(a,c,L), onLine(b,L)], Not(bet(a,b,c)) )
pc.have( Not( And(bet(a,b,c), onLine(b,L))  ), 'P3' )
pc.have( Not(bet(a,b,c)) )
pc.qed('nbet_if_side')

1. sameside(a, c, L)
2. onLine(b, L)    / Not(bet(a, b, c))
--------------------
3. Not(And(bet(a, b, c), onLine(b, L)))	[P3]
4. Not(bet(a, b, c))	[None]
🔥 Success! You proved Not(bet(a, b, c)) is true.
💾 Registered nbet_if_side as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, L],
       Implies(And(sameside(a, c, L), onLine(b, L)),
               Not(bet(a, b, c))))


True

In [401]:
# Lemma crossbar: If p is inside angle abc, then ray bp intersects ac at constructed point e between a and c.
pc = Proof( [ interior(p,a,b,c) ], And( sameray(p,e,b), bet(a,e,c) ) )
pc.have( diffside(a,c,bp), 'diff_if_int' )
pc.have( interLL(bp,ac), 'I1' )
pc.construct('CI1', e, [bp,ac])
pc.have( Distinct(a,b,c), 'ne_if_diff' )
pc.have( And( onLine(e,bp), onLine(e,ac), bet(a,e,c) ), 'P4' )
pc.have( p!=b, 'S3' )
pc.have( Not(onLine(a,bc)), 'S3' )
pc.have( Not(onLine(c,ab)), 'S3' )
pc.have( triangle(a,b,c), 'on_iff_coll' )
pc.have( ae == ac, 'G1' )
pc.have( ab != ac, 'ne_if_tri' )
pc.have( ab != ae, 'ne_if_tri' )
pc.have( e!=b, 'ne_if_tri')
pc.have( Not(onLine(e,ab)), 'G1' )
pc.have( sameside(e,c,ab), 'P2' )
pc.have( sameside(p,e,ab), 'S4')
pc.have( Not(onLine(p,ab)), 'S3' )
pc.have( Not(bet(p,b,e)), 'nbet_if_side' )
pc.have( sameray(p,e,b), 'B6' )
pc.have( And( sameray(p,e,b), bet(a,e,c) ) )
pc.qed('crossbar')

1. And(sameside(p, a, line(b, c)), sameside(p, c, line(a, b)))    / And(And(p != b,
        e != b,
        Or(p == e, bet(b, p, e), bet(b, e, p))),
    bet(a, e, c))
--------------------
2. And(Not(onLine(a, line(b, p))),
    Not(onLine(c, line(b, p))),
    Not(sameside(a, c, line(b, p))))	[diff_if_int]
3. interLL(line(b, p), line(a, c))	[I1]
4. Constructed e where And(onLine(e, line(b, p)), onLine(e, line(a, c))) 	[CI1]
5. Distinct(a, b, c)	[ne_if_diff]
6. And(onLine(e, line(b, p)),
    onLine(e, line(a, c)),
    bet(a, e, c))	[P4]
7. p != b	[S3]
8. Not(onLine(a, line(b, c)))	[S3]
9. Not(onLine(c, line(a, b)))	[S3]
10. Not(coll(a, b, c))	[on_iff_coll]
11. line(a, e) == line(a, c)	[G1]
12. line(a, b) != line(a, c)	[ne_if_tri]
13. line(a, b) != line(a, e)	[ne_if_tri]
14. e != b	[ne_if_tri]
15. Not(onLine(e, line(a, b)))	[G1]
16. sameside(e, c, line(a, b))	[P2]
17. sameside(p, e, line(a, b))	[S4]
18. Not(onLine(p, line(a, b)))	[S3]
19. Not(bet(p, b, e))	[nbet_if_side]
20. And(p != b, e 

True

In [402]:
# Lemma crossbar(Triangle ver.): If p is inside triangle abc, then construct e s.t. bp intersects ac at point e such that a-e-c and b-p-e
pc = Proof( [ insideT(p,a,b,c) ], And( bet(a,e,c), bet(b,p,e) ) )
pc.construct('crossbar', e, [])
pc.have( Distinct(p,a,b,c), 'ne_if_ins' )
pc.have(sameside(p,b,ac))
pc.have( Not(onLine(b,ac)), 'S3' )
pc.have( Not(onLine(p,ac)), 'S3' )
pc.have( onLine(e,ac), 'B3' )
pc.have( Distinct(b,p,e), 'G1' )
pc.have( Or( bet(b,p,e), bet(b,e,p), bet(p,b,e)  ), 'B6' )  
pc.have( Or( bet(b,p,e), bet(b,e,p) )  )
pc.have( Not(bet(b,e,p)), 'P3')
pc.have( And( bet(a,e,c), bet(b,p,e) ))
pc.qed('crossbarT')

1. And(sameside(p, a, line(b, c)),
    sameside(p, b, line(a, c)),
    sameside(p, c, line(a, b)))    / And(bet(a, e, c), bet(b, p, e))
--------------------
2. Constructed e where And(And(p != b,
        e != b,
        Or(p == e, bet(b, p, e), bet(b, e, p))),
    bet(a, e, c)) 	[crossbar]
3. Distinct(p, a, b, c)	[ne_if_ins]
4. sameside(p, b, line(a, c))	[None]
5. Not(onLine(b, line(a, c)))	[S3]
6. Not(onLine(p, line(a, c)))	[S3]
7. onLine(e, line(a, c))	[B3]
8. Distinct(b, p, e)	[G1]
9. Or(bet(b, p, e), bet(b, e, p), bet(p, b, e))	[B6]
10. Or(bet(b, p, e), bet(b, e, p))	[None]
11. Not(bet(b, e, p))	[P3]
12. And(bet(a, e, c), bet(b, p, e))	[None]
🔥 Success! You proved And(bet(a, e, c), bet(b, p, e)) is true.
💾 Added Construction Rule crossbarT: [lambda a, b, c, p: And(sameside(p, a, line(b, c)),
    sameside(p, b, line(a, c)),
    sameside(p, c, line(a, b))), lambda e, a, b, c, p: And(bet(a, e, c), bet(b, p, e))] into CONSTRUCTION RULES.


True

In [405]:
# Lemma int_if_bet: If bet(a,b,c), d ~on ac, then b inside angle adc.
pc = Proof( [bet(a,b,c), Not(onLine(d,ac))], interior(b,a,d,c) )
pc.have( And( ab == ac, bc == ac), 'eq_if_bet' )
pc.have( Not(onLine(b,dc)), 'G1' )
pc.have( sameside(b, a, dc), 'P2')
pc.have( Not(onLine(b,ad)), 'G1' )
pc.have( sameside(b, c, line(a, d)), 'P2' )
pc.have( interior(b,a,d,c) )
pc.qed('int_if_bet')

1. bet(a, b, c)
2. Not(onLine(d, line(a, c)))    / And(sameside(b, a, line(d, c)), sameside(b, c, line(a, d)))
--------------------
3. And(line(a, b) == line(a, c), line(b, c) == line(a, c))	[eq_if_bet]
4. Not(onLine(b, line(d, c)))	[G1]
5. sameside(b, a, line(d, c))	[P2]
6. Not(onLine(b, line(a, d)))	[G1]
7. sameside(b, c, line(a, d))	[P2]
8. And(sameside(b, a, line(d, c)), sameside(b, c, line(a, d)))	[None]
🔥 Success! You proved And(sameside(b, a, line(d, c)), sameside(b, c, line(a, d))) is true.
💾 Registered int_if_bet as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, d],
       Implies(And(bet(a, b, c), Not(onLine(d, line(a, c)))),
               And(sameside(b, a, line(d, c)),
                   sameside(b, c, line(a, d)))))


True

In [406]:
# Lemma bet_if_int: If d inside angle abc, d on ac, then d is between a and c.
pc = Proof( [interior(d,a,b,c), onLine(d,ac)], bet(a,d,c) )
pc.have( Not(onLine(d,ab)), 'S3' )
pc.have( sameside(c,d,ab) )
pc.have( bet(a,d,c) )
pc.have( Distinct(a,b,c,d), 'ne_if_int' )
pc.have( Or( bet(a,c,d), bet(c,a,d), bet(a,d,c) ), 'B6' )
pc.have( Not(bet(a,c,d)), 'P3' )
pc.have( Not(bet(c,a,d)), 'P3' )
pc.have( bet(a,d,c) )
pc.qed('bet_if_int')

1. And(sameside(d, a, line(b, c)), sameside(d, c, line(a, b)))
2. onLine(d, line(a, c))    / bet(a, d, c)
--------------------
3. Not(onLine(d, line(a, b)))	[S3]
4. sameside(c, d, line(a, b))	[None]
--> #5 have(bet(a, d, c), None): Sorry, can't infer bet(a, d, c) from None.
5. Distinct(a, b, c, d)	[ne_if_int]
6. Or(bet(a, c, d), bet(c, a, d), bet(a, d, c))	[B6]
7. Not(bet(a, c, d))	[P3]
8. Not(bet(c, a, d))	[P3]
9. bet(a, d, c)	[None]
🔥 Success! You proved bet(a, d, c) is true.
💾 Registered bet_if_int as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, d],
       Implies(And(And(sameside(d, a, line(b, c)),
                       sameside(d, c, line(a, b))),
                   onLine(d, line(a, c))),
               bet(a, d, c)))


True

In [408]:
# Lemma eq_if_line_ne: Two non-equal intersecting lines intersect at exactly one point.
pc = Proof( [ L != M, onLine(a,L), onLine(a,M), onLine(b,L), onLine(b,M) ], a == b )
pc.have( Or(a==b, a!=b) )
pc.cases( Or(a==b, a!=b) )
pc.close()
pc.have( L == M, 'G1' )
pc.have( a==b )
pc.close()
pc.qed('eq_if_line_ne')

1. L != M
2. onLine(a, L)
3. onLine(a, M)
4. onLine(b, L)
5. onLine(b, M)    / a == b
--------------------
6. Or(a == b, a != b)	[None]
7.1. Assuming case a == b. Try to reach the goal.
📁 Case closed! a == b's goal reached.
7.2. Assuming case a != b. Try to reach the goal.
8. L == M	[G1]
9. a == b	[None]
📁 Case closed! a != b's goal reached.
All cases closed! a == b have been added to context.
🔥 Success! You proved a == b is true.
💾 Registered eq_if_line_ne as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, L, M],
       Implies(And(L != M,
                   onLine(a, L),
                   onLine(a, M),
                   onLine(b, L),
                   onLine(b, M)),
               a == b))


True

In [409]:
# Lemma ins_extended_if_ins: Given triangle a b c with d inside, if extend ac to point e, then d is inside triangle abe.
pc = Proof( [insideT(d,a,b,c), bet(a,c,e) ], insideT(d,a,b,e) )
pc.have( triangle(a,b,c), 'tri_if_ins' )
pc.have( ac == ae, 'eq_if_bet' )
pc.have( sameside(b,d,ae) )    # Cond 1
pc.have( Not(onLine(c,ab)), 'S3' )
pc.have( sameside(c,e,ab), 'P2' )
pc.have( sameside(d,e,ab), 'S4' )   # Cond 2
pc.have( Not(onLine(c,be)), 'G1' )
pc.have( sameside(a,c,be), 'P2')
pc.have( Not(onLine(a,bc)), 'S3' )
pc.have( Not(onLine(e,bc)), 'G1' )
# 1. Line bc passes through c. Since bet(a, c, e), a and e are on opposite sides of bc.
pc.have( diffside(a, e, bc), 'P3' )
# 2. Since a and d are on the same side of bc (from insideT), d and e are on opposite sides of bc.
pc.have( Not(onLine(d,bc)), 'S3' )
#pc.have( And(Not(onLine(a,bc)), Not(onLine(e,bc)), Not(onLine(d,bc))) )
pc.have( Xor(sameside(a,d,bc),sameside(e,d,bc)), 'S5' )
pc.have( sameside(a, d, bc) )
pc.have( Not(sameside(e,d,bc)) )
pc.have( diffside(e,d,bc) )
pc.have( interior(c, a, b, e), 'int_if_bet' )
pc.construct('crossbarT', f, [a,b,c,d])
pc.have( bet(a,f,e), 'B4')

pc.have( Not(onLine(b,ac)), 'S3' )
pc.have( interior(f,a,b,e), 'int_if_bet' )
pc.have( sameside(a,f,be), 'P2' )
pc.have( bet(b,d,f))   # From crossbarT
pc.construct('LC1', L, [a,c])
pc.have( onLine(e,L), 'G1' )
pc.construct('LC1', M, [b,f])
pc.have( e != f )
pc.have( Distinct(b,d,e) )
pc.have( bd == bf, 'eq_if_bet' )
pc.have( bf == M, 'G1' )
pc.have( ae == L, 'G1' )
pc.have( And(onLine(f,L), onLine(f,M)) , 'B3' )
# Trying to show it is not possible that d lies on be
pc.have( Implies( onLine(d,be), bd == be ), 'G1' )
pc.have( Implies( bd == be,  be == M ), 'G1' )
pc.have( Implies( be == M,  onLine(e,M) ), 'G1' )
pc.have( Implies( onLine(d,be), And(onLine(e,M), onLine(e,L), onLine(f,L), onLine(f,M) )  ) )
pc.have( Implies( onLine(d,be), e == f  ), 'eq_if_line_ne' )
pc.have( Not(onLine(d,be)) )

pc.have( sameside(d,f,be), 'P2' )
pc.have( sameside(a, d, be),  'S4' )  # Cond 3
pc.have( insideT(d,a,b,e))
pc.qed('ins_extended_if_ins')

1. And(sameside(d, a, line(b, c)),
    sameside(d, b, line(a, c)),
    sameside(d, c, line(a, b)))
2. bet(a, c, e)    / And(sameside(d, a, line(b, e)),
    sameside(d, b, line(a, e)),
    sameside(d, e, line(a, b)))
--------------------
3. Not(coll(a, b, c))	[tri_if_ins]
4. line(a, c) == line(a, e)	[eq_if_bet]
5. sameside(b, d, line(a, e))	[None]
6. Not(onLine(c, line(a, b)))	[S3]
7. sameside(c, e, line(a, b))	[P2]
8. sameside(d, e, line(a, b))	[S4]
9. Not(onLine(c, line(b, e)))	[G1]
10. sameside(a, c, line(b, e))	[P2]
11. Not(onLine(a, line(b, c)))	[S3]
12. Not(onLine(e, line(b, c)))	[G1]
13. And(Not(onLine(a, line(b, c))),
    Not(onLine(e, line(b, c))),
    Not(sameside(a, e, line(b, c))))	[P3]
14. Not(onLine(d, line(b, c)))	[S3]
15. Xor(sameside(a, d, line(b, c)), sameside(e, d, line(b, c)))	[S5]
16. sameside(a, d, line(b, c))	[None]
17. Not(sameside(e, d, line(b, c)))	[None]
18. And(Not(onLine(e, line(b, c))),
    Not(onLine(d, line(b, c))),
    Not(sameside(e, d, line(b, c))))	[N

True

In [44]:
# Lemma plane_sep: Given a not on L, either p a sameside L or p on L or p a diffside L.
pc = Proof( [ points(a,p), Not(onLine(a,L))], Or( sameside(p,a,L), onLine(p,L), diffside(p,a,L) ) )
pc.have( Or( sameside(p,a,L), onLine(p,L), diffside(p,a,L) ) )
pc.qed('plane_sep')

1. Or(a != p, Not(a != p))
2. Not(onLine(a, L))    / Or(sameside(p, a, L),
   onLine(p, L),
   And(Not(onLine(p, L)),
       Not(onLine(a, L)),
       Not(sameside(p, a, L))))
--------------------
3. Or(sameside(p, a, L),
   onLine(p, L),
   And(Not(onLine(p, L)),
       Not(onLine(a, L)),
       Not(sameside(p, a, L))))	[None]
🔥 Success! You proved Or(sameside(p, a, L),
   onLine(p, L),
   And(Not(onLine(p, L)),
       Not(onLine(a, L)),
       Not(sameside(p, a, L)))) is true.
💾 Registered plane_sep as a Universal Theorem in AXIOMS!
   Form: ForAll([a, p, L],
       Implies(And(Or(a != p, Not(a != p)),
                   Not(onLine(a, L))),
               Or(sameside(p, a, L),
                  onLine(p, L),
                  And(Not(onLine(p, L)),
                      Not(onLine(a, L)),
                      Not(sameside(p, a, L))))))


True

In [413]:
# Aux 7a: Given triangle a b c with d inside, if c d sameside ab, ac = ad, cb = db, prove c = d (by contradiction)
pc = Proof( [ insideT(d,a,b,c), sameside(c,d,ab), 
           Segment(a,c)==Segment(a,d), Segment(c,b)==Segment(d,b) ], c == d )
pc.have( triangle(a,b,c), 'tri_if_ins' )
pc.have( c != d, 'ne_if_ins' )
pc.have( Not(onLine(d,ab)), 'S3' )
pc.have( d!=a )
pc.have( Distinct(a,b,c,d) )
pc.have( Angle(a,c,d) == Angle(a,d,c), 'E5' )
pc.have( Angle(b,c,d) == Angle(b,d,c), 'E5' )
pc.construct('CP4', e, [a,c,ac])
pc.construct('LC1', ad, [a,d])
pc.construct('CP4', f, [a,d,ad] )
pc.have( Angle(d,c,e) == Angle(c,d,f), 'E5a' )
pc.have( Angle(c,d,b) == Angle(d,c,b), 'E5' )
pc.have( onLine(e, line(c,e)) )
pc.have( ac == ce, 'G1' )
## Trying to get Angle(d,c,b) < Angle(d,c,e)
pc.have( sameside(d,b,ce) )
pc.have( bet(a,c,e) )
pc.have( Not(sameside(a,e,bc)), 'P3' )
pc.have( Implies(onLine(e, bc), be==bc), 'G1' )
pc.have( Not(onLine(a,bc)), 'S3' )
pc.have( diffside(a, e, bc), 'Tdiff' )
pc.have( sameside(a,d, bc) )
pc.have( Not(onLine(d, bc)), 'S3' )
pc.have( diffside(d, e, bc), 'S4' )
pc.have( Not(onLine(b, ec)), 'S3' )
pc.have( Not(onLine(d, ec)), 'S3' )
pc.have( insideT(d,a,b,e), 'ins_extended_if_ins' )
pc.have( interior(d,a,e,b) )
pc.have( sameray(a,c,e), 'sameray_if_bet')
#pc.have( diffside(a,b,de), 'diff_if_int' )
pc.have( interior(d,c,e,b), 'int_sameray_if_int' )
pc.have( diffside(b,c,de), 'diff_if_int' )
#pc.have(  Implies(And(diffside(b,c,de), diffside(d,e,bc)), And(sameside(b,d,ce), sameside(b,e,cd)) ), 'diffdiff' )
pc.have( sameside(b,e,cd), 'diffdiff')
pc.have( interior(b,d,c,e) )
# pc.have( dc != ce, 'ne_if_int' )
# pc.have( Not(onLine(b, dc)), 'S3' )
# pc.have(Not(onLine(b, ce)), 'G1')
# pc.have( And(onLine(c, dc), onLine(c, ce), onLine(d, dc), onLine(e, ce), 
#                              c != d, c != e, Not(onLine(b, dc)), Not(onLine(b, ce)), dc != ce ) )
# pc.have( Angle(d, c, e) == Angle(d, c, b) + Angle(b, c, e), 'DA2' )
# pc.have(Not(bet(e, c, b)), 'B2')
# pc.have( Not(bet(b,c,e)), 'B1' )
# pc.have( Angle(b, c, e) > 0, 'DA1' )
pc.have( Angle(d,c,b) < Angle(d,c,e), 'lt_if_int' )
pc.have(Not(sameside(a, f, bd)), 'P3')
#pc.have(sameside(d, f, ab), rule='P2')
#pc.have(sameside(c, f, ab), rule='S4') # Links c and f since sameside(c,d,ab) is a hypothesis
# 2. Anchor point f relative to line ac (or ce) using betweenness bet(a,d,f)
pc.have( Not(onLine(d,ac)), 'S3' )
pc.have(sameside(d, f, ac), 'P2')
pc.have(sameside(b, f, ac), 'S4') # Links b and f since insideT(d,a,b,c) implies sameside(b,d,ac)
pc.have( interior(d,a,b,c) )
pc.have( diffside(a,c, bd), 'diff_if_int' )
pc.have( Not(onLine(f,bd)), 'G1' )
pc.have( sameside(c,f,bd), 'S5')
pc.have( diffside(a,b, dc), 'diff_if_int' )
pc.have( Not(onLine(f,dc)), 'G1' )
pc.have(Not(sameside(a, f, dc)), 'P3')
pc.have( sameside(b,f,dc), 'S5')
pc.have( interior(f,b,d,c) )
pc.have( Angle(c,d,f) < Angle(c,d,b), 'lt_if_int' )

pc.have( And( Angle(d,c,b) < Angle(d,c,e) ) )
pc.have( Angle(d,c,e) == Angle(c,d,f) )
pc.have( Angle(d,c,b) < Angle(c,d,b), 'L4' )
pc.have( Angle(d,c,b) == Angle(c,d,b) )
pc.have( And( Angle(d,c,b) < Angle(c,d,b), Angle(d,c,b) == Angle(c,d,b),  c == d  ) )
pc.have( c == d )
pc.qed('A7a')

1. And(sameside(d, a, line(b, c)),
    sameside(d, b, line(a, c)),
    sameside(d, c, line(a, b)))
2. sameside(c, d, line(a, b))
3. Segment(a, c) == Segment(a, d)
4. Segment(c, b) == Segment(d, b)    / c == d
--------------------
5. Not(coll(a, b, c))	[tri_if_ins]
6. c != d	[ne_if_ins]
7. Not(onLine(d, line(a, b)))	[S3]
8. d != a	[None]
9. Distinct(a, b, c, d)	[None]
10. Angle(a, c, d) == Angle(a, d, c)	[E5]
11. Angle(b, c, d) == Angle(b, d, c)	[E5]
12. Constructed e where And(onLine(e, line(a, c)), bet(a, c, e)) 	[CP4]
13. Constructed line(a, d) where And(onLine(a, line(a, d)), onLine(d, line(a, d))) 	[LC1]
14. Constructed f where And(onLine(f, line(a, d)), bet(a, d, f)) 	[CP4]
15. Angle(d, c, e) == Angle(c, d, f)	[E5a]
16. Angle(c, d, b) == Angle(d, c, b)	[E5]
17. onLine(e, line(c, e))	[None]
18. line(a, c) == line(c, e)	[G1]
19. sameside(d, b, line(c, e))	[None]
20. bet(a, c, e)	[None]
21. Not(sameside(a, e, line(b, c)))	[P3]
22. Implies(onLine(e, line(b, c)), line(b, e) == line(b, 

True

In [414]:
# Aux 7b: Given triangle a b c with d a diffside bc, if d inside angle bac, ac = ad, cb = db, prove c = d (by contradiction)
pc = Proof( [triangle(a,b,c), interior(d,b,a,c), diffside(a,d,bc), 
           Segment(a,c)==Segment(a,d), Segment(c,b)==Segment(d,b) ], c == d )
pc.have( a!=d, 'S1' )
pc.have( Distinct(a, c, d) )
pc.have( Angle(a,c,d) == Angle(a,d,c), 'E5' )
pc.have( Angle(b,c,d) == Angle(b,d,c), 'E5' )
pc.construct('crossbar', e, [b,a,c,d])
pc.have( diffside(b,c,ad), 'diff_if_int' )
pc.have( sameside(a, b, dc), 'diffdiff' )
pc.have( sameside(a, c, bd),'diffdiff' )
pc.have( interior(a,b,d,c) )
pc.have( Angle(b,d,c) > Angle(a,d,c), 'lt_if_int' )
pc.have( interior(b,a,c,d) )
pc.have( Angle(a,c,d) > Angle(b,c,d), 'lt_if_int' )
pc.have( Angle(b,d,c) > Angle(b,d,c) )
pc.have( c == d )
pc.qed('A7b')

1. Not(coll(a, b, c))
2. And(sameside(d, b, line(a, c)), sameside(d, c, line(b, a)))
3. And(Not(onLine(a, line(b, c))),
    Not(onLine(d, line(b, c))),
    Not(sameside(a, d, line(b, c))))
4. Segment(a, c) == Segment(a, d)
5. Segment(c, b) == Segment(d, b)    / c == d
--------------------
6. a != d	[S1]
7. Distinct(a, c, d)	[None]
8. Angle(a, c, d) == Angle(a, d, c)	[E5]
9. Angle(b, c, d) == Angle(b, d, c)	[E5]
10. Constructed e where And(And(d != a,
        e != a,
        Or(d == e, bet(a, d, e), bet(a, e, d))),
    bet(b, e, c)) 	[crossbar]
11. And(Not(onLine(b, line(a, d))),
    Not(onLine(c, line(a, d))),
    Not(sameside(b, c, line(a, d))))	[diff_if_int]
12. sameside(a, b, line(d, c))	[diffdiff]
13. sameside(a, c, line(b, d))	[diffdiff]
14. And(sameside(a, b, line(d, c)), sameside(a, c, line(b, d)))	[None]
15. Angle(b, d, c) > Angle(a, d, c)	[lt_if_int]
16. And(sameside(b, a, line(c, d)), sameside(b, d, line(a, c)))	[None]
17. Angle(a, c, d) > Angle(b, c, d)	[lt_if_int]
18. Angle

True

In [415]:
# Aux 7c: Given triangle a b c, if d inside angle bac, ac = ad, cb = db, prove c = d (by contradiction)
pc = Proof( [triangle(a,b,c), interior(d,b,a,c), Segment(a,c)==Segment(a,d), Segment(c,b)==Segment(d,b) ], c == d )
pc.have( Not(onLine(a,bc)), 'non_if_tri' )
pc.have( Or( sameside(a,d,bc),  diffside(a,d,bc), onLine(d,bc) ), 'P3'  )
pc.cases( Or( sameside(a,d,bc),  diffside(a,d,bc), onLine(d,bc) ) )
pc.have( insideT(d,a,b,c) )
pc.have( c == d, 'A7a' )
pc.close()
pc.have( c==d, 'A7b' )
pc.close()
pc.have( bet(b,d,c), 'bet_if_int' )
pc.have( Segment(c,b) == Segment(d,b) + Segment(c,d), 'DS1' )
pc.have( Segment(c,d) > 0, 'M1' ) 
pc.have( Segment(c,b) != Segment(d,b) )
pc.have( c==d )
pc.close()
pc.qed('A7c')

1. Not(coll(a, b, c))
2. And(sameside(d, b, line(a, c)), sameside(d, c, line(b, a)))
3. Segment(a, c) == Segment(a, d)
4. Segment(c, b) == Segment(d, b)    / c == d
--------------------
5. Not(onLine(a, line(b, c)))	[non_if_tri]
6. Or(sameside(a, d, line(b, c)),
   And(Not(onLine(a, line(b, c))),
       Not(onLine(d, line(b, c))),
       Not(sameside(a, d, line(b, c)))),
   onLine(d, line(b, c)))	[P3]
7.1. Assuming case sameside(a, d, line(b, c)). Try to reach the goal.
8. And(sameside(d, a, line(b, c)),
    sameside(d, b, line(a, c)),
    sameside(d, c, line(a, b)))	[None]
9. c == d	[A7a]
📁 Case closed! sameside(a, d, line(b, c))'s goal reached.
7.2. Assuming case And(Not(onLine(a, line(b, c))),
    Not(onLine(d, line(b, c))),
    Not(sameside(a, d, line(b, c)))). Try to reach the goal.
8. c == d	[A7b]
📁 Case closed! And(Not(onLine(a, line(b, c))),
    Not(onLine(d, line(b, c))),
    Not(sameside(a, d, line(b, c))))'s goal reached.
7.3. Assuming case onLine(d, line(b, c)). Try to reac

True

In [416]:
# Aux 7d: Given triangle a b c, if c d sameside ab, bet(a,c,d), ac = ad, cb = db, prove c = d (by contradiction)
pc = Proof( [triangle(a,b,c), sameside(c,d,ab), Or(bet(a,c,d), bet(a,d,c)) , Segment(a,c)==Segment(a,d), Segment(c,b)==Segment(d,b) ], c == d )
pc.have( Segment(a,d) == Segment(a,c) + Segment(c,d), 'DS1' )
pc.have( Segment(a,c) != Segment(a,d), 'M1' )
pc.have( c==d )
pc.qed('A7d')

1. Not(coll(a, b, c))
2. sameside(c, d, line(a, b))
3. Or(bet(a, c, d), bet(a, d, c))
4. Segment(a, c) == Segment(a, d)
5. Segment(c, b) == Segment(d, b)    / c == d
--------------------
6. Segment(a, d) == Segment(a, c) + Segment(c, d)	[DS1]
7. Segment(a, c) != Segment(a, d)	[M1]
8. c == d	[None]
🔥 Success! You proved c == d is true.
💾 Registered A7d as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, d],
       Implies(And(Not(coll(a, b, c)),
                   sameside(c, d, line(a, b)),
                   Or(bet(a, c, d), bet(a, d, c)),
                   Segment(a, c) == Segment(a, d),
                   Segment(c, b) == Segment(d, b)),
               c == d))


True

In [417]:
# Proposition 7: Given triangle a b c, if c d sameside ab, ac = ad, cb = db, prove c = d.
pc = Proof( [triangle(a,b,c), sameside(c,d,ab), 
           Segment(a,c)==Segment(a,d), Segment(c,b)==Segment(d,b) ], c == d )
pc.have( Not(onLine(d,ab)), 'S3' )
pc.have( Not(onLine(b,ac)), 'non_if_tri' )
pc.have( Not(onLine(a,bc)), 'non_if_tri' )
pc.have( d!=a )
pc.have( Angle(a,c,d) == Angle(a,d,c), 'E5' )
pc.have( Angle(b,c,d) == Angle(b,d,c), 'E5' )
pc.have( Or( sameside(d,b,ac), onLine(d,ac), diffside(d,b,ac) ), 'plane_sep' )
pc.cases( Or( sameside(d,b,ac), onLine(d,ac), diffside(d,b,ac) ))   # Consider each case and show contradiction
pc.have( interior(d,b,a,c) )
pc.have( c==d, 'A7c' )
pc.close()
pc.have( coll(c,d,a), 'L4' )
pc.have( sameray(c,d,a), 'R2a' )
pc.have( Or(c==d, bet(a,c,d), bet(a,d,c)) )
pc.have( c==d, 'A7d' )
pc.close()
pc.have( Or( sameside(d,a,bc), onLine(d,bc), diffside(d,a,bc) ), 'plane_sep' )
pc.cases( Or( sameside(d,a,bc), onLine(d,bc), diffside(d,a,bc) ) )
pc.have( interior(d,a,b,c) )
pc.have( c==d, 'A7c' )
pc.close()
pc.have( coll(c,d,b), 'L4' )
pc.have( sameray(c,d,b), 'R2a' )
pc.have( c==d, 'A7d' )
pc.close()
pc.have( insideT(c,a,b,d), 'sameside_if_samediff' )
pc.have( c==d, 'A7a' )
pc.close()
pc.close()
pc.qed('E7')

1. Not(coll(a, b, c))
2. sameside(c, d, line(a, b))
3. Segment(a, c) == Segment(a, d)
4. Segment(c, b) == Segment(d, b)    / c == d
--------------------
5. Not(onLine(d, line(a, b)))	[S3]
6. Not(onLine(b, line(a, c)))	[non_if_tri]
7. Not(onLine(a, line(b, c)))	[non_if_tri]
8. d != a	[None]
9. Angle(a, c, d) == Angle(a, d, c)	[E5]
10. Angle(b, c, d) == Angle(b, d, c)	[E5]
11. Or(sameside(d, b, line(a, c)),
   onLine(d, line(a, c)),
   And(Not(onLine(d, line(a, c))),
       Not(onLine(b, line(a, c))),
       Not(sameside(d, b, line(a, c)))))	[plane_sep]
12.1. Assuming case sameside(d, b, line(a, c)). Try to reach the goal.
13. And(sameside(d, b, line(a, c)), sameside(d, c, line(b, a)))	[None]
14. c == d	[A7c]
📁 Case closed! sameside(d, b, line(a, c))'s goal reached.
12.2. Assuming case onLine(d, line(a, c)). Try to reach the goal.
13. coll(c, d, a)	[L4]
14. And(c != a, d != a, Or(c == d, bet(a, c, d), bet(a, d, c)))	[R2a]
15. Or(c == d, bet(a, c, d), bet(a, d, c))	[None]
16. c == d	[A7d]

True

In [418]:
# Proposition 8a: If ab = de, ac = df, bc = ef, then angle bac = angle edf
pc = Proof( [triangle(a,b,c), triangle(d,e,f), Segment(a,b) == Segment(d,e), Segment(a,c) == Segment(d,f), 
                      Segment(b,c) == Segment(e,f)], Angle(b,a,c) == Angle(e,d,f) )
pc.have( Not(onLine(d,ef)), 'non_if_tri' )
pc.construct('SP', [y,z,x], [b,c,a,e,f,d]  )
pc.have( Segment(e,z) == Segment(e,f) )
pc.have( Or(z == f, bet(e, z, f), bet(e, f, z)) )
pc.have( Implies( bet(e, z, f), Segment(e,f)==Segment(e,z) + Segment(z,f) ), 'DS1' )
pc.have( Implies( bet(e, z, f), z == f ), 'M1' )
pc.have( Implies( bet(e, f, z), Segment(e,z)==Segment(e,f) + Segment(z,f) ), 'DS1' )
pc.have( Implies( bet(e, f, z), z == f ), 'M1' )
pc.have( z==f, 'M1' )
pc.have( Implies( And(triangle(d,e,f), sameside(x,d,ef), Segment(e,x)==Segment(e,d), Segment(f,x)==Segment(f,d)), x==d ), 'E7' )
pc.have( Segment(e,x)==Segment(e,d) )
pc.have( Segment(f,x)==Segment(f,d) )
pc.have( d==x)
pc.have(  Angle(b,a,c) == Angle(e,d,f) )
pc.qed('E8a')

1. Not(coll(a, b, c))
2. Not(coll(d, e, f))
3. Segment(a, b) == Segment(d, e)
4. Segment(a, c) == Segment(d, f)
5. Segment(b, c) == Segment(e, f)    / Angle(b, a, c) == Angle(e, d, f)
--------------------
6. Not(onLine(d, line(e, f)))	[non_if_tri]
7. Constructed [y, z, x] where And(y == e,
    And(z != e,
        f != e,
        Or(z == f, bet(e, z, f), bet(e, f, z))),
    sameside(x, d, line(e, f)),
    cong(y, z, x, b, c, a)) 	[SP]
8. Segment(e, z) == Segment(e, f)	[None]
9. Or(z == f, bet(e, z, f), bet(e, f, z))	[None]
10. Implies(bet(e, z, f),
        Segment(e, f) == Segment(e, z) + Segment(z, f))	[DS1]
11. Implies(bet(e, z, f), z == f)	[M1]
12. Implies(bet(e, f, z),
        Segment(e, z) == Segment(e, f) + Segment(z, f))	[DS1]
13. Implies(bet(e, f, z), z == f)	[M1]
14. z == f	[M1]
15. Implies(And(Not(coll(d, e, f)),
            sameside(x, d, line(e, f)),
            Segment(e, x) == Segment(e, d),
            Segment(f, x) == Segment(f, d)),
        x == d)	[E7]
16. Segment(e, x

True

In [419]:
# Proposition 8: If ab = de, ac = df, bc = ef, then triangle abc congreunt to triangle def
pc = Proof( [triangle(a,b,c), triangle(d,e,f), Segment(a,b) == Segment(d,e), Segment(a,c) == Segment(d,f), 
                      Segment(b,c) == Segment(e,f)], cong(a,b,c,d,e,f) )
pc.have( Angle(b,a,c) == Angle(e,d,f), 'E8a' )
pc.have( cong(b,a,c,e,d,f), 'E4' )
pc.have( cong(a,b,c,d,e,f)  )
pc.qed('E8')

1. Not(coll(a, b, c))
2. Not(coll(d, e, f))
3. Segment(a, b) == Segment(d, e)
4. Segment(a, c) == Segment(d, f)
5. Segment(b, c) == Segment(e, f)    / cong(a, b, c, d, e, f)
--------------------
6. Angle(b, a, c) == Angle(e, d, f)	[E8a]
7. cong(b, a, c, e, d, f)	[E4]
8. cong(a, b, c, d, e, f)	[None]
🔥 Success! You proved cong(a, b, c, d, e, f) is true.
💾 Registered E8 as a Universal Theorem in AXIOMS!
   Form: ForAll([a, b, c, d, e, f],
       Implies(And(Not(coll(a, b, c)),
                   Not(coll(d, e, f)),
                   Segment(a, b) == Segment(d, e),
                   Segment(a, c) == Segment(d, f),
                   Segment(b, c) == Segment(e, f)),
               cong(a, b, c, d, e, f)))


True

In [423]:
# Lemma sameray_from_ins: Given a inside α, b!=a, construct e such that e b sameray a.
pc = Proof( [inside(a,α), a!=b], And( onCircle(e,α), sameray(e,b,a) ) )
pc.construct('LC1', L, [a,b])
pc.construct('CI5', d, [a,b,L,α])
pc.have( interLC(L,α), 'I3' )
pc.construct('CI3', [f,g], [L,α])
pc.have( bet(f,a,g), 'C1' )

1. inside(a, α)
2. a != b    / And(onCircle(e, α),
    And(e != a,
        b != a,
        Or(e == b, bet(a, e, b), bet(a, b, e))))
--------------------
3. Constructed L where And(onLine(a, L), onLine(b, L)) 	[LC1]
4. Constructed d where And(onCircle(d, α), onLine(d, L), bet(d, a, b)) 	[CI5]
5. interLC(L, α)	[I3]
6. Constructed [f, g] where And(onCircle(f, α),
    onLine(f, L),
    onCircle(g, α),
    onLine(g, L),
    f != g) 	[CI3]
7. bet(f, a, g)	[C1]


In [421]:
pc.have( Or( insideT(d,a,b,c), Not(insideT(d,a,b,c)) ) )
pc.cases( Or( insideT(d,a,b,c), Not(insideT(d,a,b,c)) ) )

6. Or(And(sameside(d, a, line(b, c)),
       sameside(d, b, line(a, c)),
       sameside(d, c, line(a, b))),
   Not(And(sameside(d, a, line(b, c)),
           sameside(d, b, line(a, c)),
           sameside(d, c, line(a, b)))))	[None]
7.1. Assuming case And(sameside(d, a, line(b, c)),
    sameside(d, b, line(a, c)),
    sameside(d, c, line(a, b))). Try to reach the goal.
